# PetFinder — Modelo Tabular
**Maestría en Ciencia de Datos — Laboratorio de Implementación**

Este notebook cubre tres etapas en secuencia:
1. **Baseline** — LightGBM con features crudas, sin ingeniería
2. **Feature Engineering iterativo** — creación y evaluación incremental de variables
3. **Tuning** — optimización de hiperparámetros con Optuna

**Métrica principal:** Cohen's Kappa Cuadrático (QWK)  
**Métricas de seguimiento:** accuracy por clase, F1 macro  
**Validación:** StratifiedKFold(n_splits=5)

# 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
import os
import re
import time
import datetime
from collections import defaultdict
import pickle
from scipy import stats
import os
import joblib

# Modelado
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import TargetEncoder
from sklearn.decomposition import PCA

# Text
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
from pysentimiento import create_analyzer
from sentence_transformers import SentenceTransformer

# Imágenes
from tqdm import tqdm
from PIL import Image
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

# Tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore', category=UserWarning, module='lightgbm')

plt.rcParams.update({
    'figure.dpi': 110,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})
sns.set_palette('Set2')
PALETTE = sns.color_palette('Set2', 5)

SEEDS = [310019, 320009, 330017, 340007, 350029]
SEED    = 42
TARGET  = 'AdoptionSpeed'
N_FOLDS = 5
np.random.seed(SEED)

print('Librerías cargadas OK')
print(f'LightGBM version: {lgb.__version__}')

In [ ]:
kaggle_input_path = '/kaggle/input/competitions/petfinder-adoption-prediction/'
local_input_path   = '../input/petfinder-adoption-prediction/'

train  = pd.read_csv(os.path.join(local_input_path, 'train/train.csv'))
breeds = pd.read_csv(os.path.join(local_input_path, 'breed_labels.csv'))
colors = pd.read_csv(os.path.join(local_input_path, 'color_labels.csv'))
states = pd.read_csv(os.path.join(local_input_path, 'state_labels.csv'))

print(f'Train: {train.shape[0]:,} filas × {train.shape[1]} columnas')
train.head(2)

---
# 1. Preprocesamiento base

Realizaremos una limpieza rápida de las variables, derivada del EDA.

- Name: para identificar mejor los animales con nombres genéricos o que en realidad corresponden a valores nulos, se imputa un valor Nan a los nombres con 2 caracteres o menos, que contenga alguna de las palabras declaradas en 'invalid_kw', que contenga algún caracter extraño, o que sea sólo numérico.
- Descripción: para identificar mejor los animales con descripciones genéricas o muy cortas e imputar valores nulos, los criterios son que tenga alguna de las palabras genéricas pertenecientes al diccionario 'generic_phrases' y que la descripción tenga menos de 40 caracteres; o que la descripción tenga menos de 10 caracteres.
- Breed: si la raza principal es igual a la secundaria, se mantiene sólo la primaria.
- Color: si el color primario es igual al secundario o terciario, se mantiene solo el primario.
- Labels: se crean variables con las etitquetas de raza, color y estado.
- Desduplicados: se quitan registros que tienen los mismos valores en todas las variables salvo PetID.

In [ ]:
df = train.copy()

# ── Limpieza de Name ──────────────────────────────────────────────────────────
invalid_kw = ['adoption','urgent','none','unknown','name','nan','n/a','yet','help']
df['Name'] = df['Name'].astype(str).str.lower().str.strip()
mask_name = (
    df['Name'].str.contains('|'.join(invalid_kw), na=False) |
    (df['Name'].str.len() <= 2) |
    df['Name'].str.match(r'^[^a-zA-Z0-9]+$') |
    df['Name'].str.isnumeric() |
    (df['Name'] == 'nan')
)
df.loc[mask_name, 'Name'] = np.nan

# ── Limpieza de Description ───────────────────────────────────────────────────
generic_phrases = ['adoption','to be adopted','looking for a home','please','available','urgent']
mask_kw    = df['Description'].str.lower().str.contains('|'.join(generic_phrases), na=False)
mask_short = df['Description'].str.len() < 40
df.loc[mask_kw & mask_short, 'Description'] = np.nan
df['Description'] = df['Description'].str.strip()
df.loc[df['Description'].str.len() < 10, 'Description'] = np.nan

# ── Limpieza de Breed y Color ───────────────────────────────────────────────────────────
df.loc[df['Breed1'] == df['Breed2'], 'Breed2'] = 0
df.loc[df['Color1'] == df['Color2'], 'Color2'] = 0
df.loc[df['Color1'] == df['Color3'], 'Color3'] = 0

# ── Merge con labels de razas ─────────────────────────────────────────────────
breeds_ = breeds.copy()
breeds_['Type']    = breeds_['Type'].astype(int)
breeds_['BreedID'] = breeds_['BreedID'].astype(int)
df['Type']   = df['Type'].astype(int)
df['Breed1'] = df['Breed1'].astype(int)
df['Breed2'] = df['Breed2'].astype(int)

df = df.merge(breeds_, left_on=['Type','Breed1'], right_on=['Type','BreedID'], how='left')
df.rename(columns={'BreedName': 'Breed1Name'}, inplace=True)
df.drop(columns=['BreedID'], inplace=True)

df = df.merge(breeds_, left_on=['Type','Breed2'], right_on=['Type','BreedID'], how='left')
df.rename(columns={'BreedName': 'Breed2Name'}, inplace=True)
df.drop(columns=['BreedID'], inplace=True)

# ── Mapeos de color y estado ──────────────────────────────────────────────────
color_map = colors.set_index('ColorID')['ColorName'].to_dict()
state_map = states.set_index('StateID')['StateName'].to_dict()
df['StateName']   = df['State'].map(state_map)
df['Color1Label'] = df['Color1'].map(color_map)

# ── Deduplicación ─────────────────────────────────────────────────────────────
df = df.sort_values(by='PetID').drop_duplicates(
    subset=df.columns.difference(['PetID']), keep='first'
)

print(f'Dataset preprocesado: {df.shape}')
df.head(2)

---
# 2. Infraestructura de evaluación

### La función `run_cv` — punto de entrada principal

Función que ejecuta todo el pipeline desde el preprocesamiento de datos hasta la ejecución del modelo.

**Parámetros siempre presentes:**
- `X` — DataFrame con las features. Puede incluir columnas auxiliares (ej. `RescuerID`, `Description`) que se usan para calcular encodings internamente pero no entran directamente al modelo si son de tipo texto o ID.
- `y` — Serie con el target.
- `params` — dict con hiperparámetros de LightGBM.

**Parámetros anti-leakage (opcionales):**

| Parámetro | Qué hace | Cuándo usarlo |
|---|---|---|
| `te_cols` | Target encoding dentro del fold | Variables de alta cardinalidad (Breed1, RescuerID) |
| `te_smoothing` | Suavizado del TE (default='auto') | Ajustar si hay pocas observaciones por categoría |
| `agg_config` | Aggregations por grupo dentro del fold | Estadísticas de comportamiento (mean_age, mean_fee por rescatista) |
| `tfidf_col` | TF-IDF + SVD sobre texto dentro del fold | Columna de descripción de texto libre |
| `n_svd` | Componentes SVD (default=15) | Reducir si hay poca varianza explicada |

In [ ]:
def qwk(y_true, y_pred):
    """Cohen's Kappa Cuadrático — métrica principal del problema."""
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

def run_cv(
    X, y, params,
    cat_features=None,
    n_folds=N_FOLDS,
    verbose=True,
    experiment_name='exp',
    seed=None,         
    te_cols=None,
    te_smoothing=50,
    agg_config=None,
    agg_smoothing=0,
    aux_cols=None,
):
    """
    Entrena LightGBM con StratifiedKFold(5) y retorna métricas OOF.

    Toda transformación que depende del target o de estadísticas globales
    se recalcula DENTRO de cada fold usando solo datos de train.
    Esto garantiza que el fold de validación nunca contamina las features.

    Parámetros
    ----------
    X : pd.DataFrame
        Features. Puede incluir columnas auxiliares para TE/agg/tfidf.
        Esas columnas también entrarán al modelo a menos que las excluyas
        explícitamente de CURRENT_FEATURES.
    y : pd.Series
        Target (AdoptionSpeed).
    params : dict
        Hiperparámetros de LightGBMClassifier.
    cat_features : list[str] | None
        Columnas categóricas para LightGBM. Si None, usa 'auto'.
    n_folds : int
        Número de folds del StratifiedKFold.
    verbose : bool
        Si True, imprime métricas por fold.
    experiment_name : str
        Nombre del experimento para el tracker.
    te_cols : list[str] | None
        Columnas sobre las que aplicar target encoding dentro del fold.
        Por cada col en te_cols, se genera una nueva columna col+'_te'.
        La columna original sigue estando disponible en X.
    te_smoothing : float | 'auto'
        Factor de suavizado del TargetEncoder de sklearn.
        Con 'auto' usa empirical Bayes. Con un float (ej: 50) equivale
        al Laplace smoothing anterior: mayor valor → encoding se acerca
        más a la media global. Usar 30-100 para columnas de alta
        cardinalidad como RescuerID.
    agg_config : dict | None
        Aggregations por grupo a calcular dentro del fold.
        Formato esperado:
            {
                'RescuerID': {
                    'resc_mean_age'  : ('Age',      'mean'),
                    'resc_mean_fee'  : ('Fee',      'mean'),
                    'resc_pct_free'  : ('IsFree',   'mean'),
                }
            }
        La clave del dict es la columna de agrupación.
        Los valores son dicts: {nombre_nueva_col: (col_fuente, función_agg)}.
        Categorías que aparecen en val pero no en train → fillna con media de train.
    agg_smoothing : float
        Factor de suavizado para las aggregations por grupo.
    aux_cols : list[str] | None
        Columnas que viajan en X para ser usadas por agg_config o tfidf_col
        pero que no son features del modelo (strings, IDs, etc.).

    Retorna
    -------
    dict con claves:
        'name'      : nombre del experimento
        'qwk_mean'  : QWK medio de los 5 folds
        'qwk_std'   : desvío estándar del QWK
        'acc_mean'  : accuracy media
        'f1_mean'   : F1 macro media
        'oof_preds' : predicciones OOF (array de largo len(y))
        'feat_imp'  : DataFrame con importancia gain y split por feature
        'fold_qwk'  : lista con QWK por fold (para diagnóstico de varianza)

    Notas sobre el TargetEncoder de sklearn
    ----------------------------------------
    - fit_transform(X_tr, y_tr) aplica cross-fitting interno (cv=5 por
      defecto) para evitar leakage dentro del set de train.
    - transform(X_val) usa los encodings del fit completo sobre X_tr,
      lo que es correcto: el fold de val nunca influye en el encoding.
    - NO se usa fit(X_tr).transform(X_tr): eso sí genera leakage
      (sklearn mismo lo advierte en la documentación).
    - target_type='continuous' fuerza una sola columna de salida por
      feature, en lugar de n_clases columnas (comportamiento multiclass).
      Esto es preferible para pasar a LightGBM con un target ordinal.
    """
    _seed = seed if seed is not None else SEED   # ← fallback a la global

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=_seed)
    fold_qwk, fold_acc, fold_f1 = [], [], []
    oof_preds = np.zeros(len(y))
    oof_proba  = np.zeros((len(y), 5))

    # Se inicializan en el primer fold cuando ya sabemos el shape final de X
    feature_names  = None
    feat_imp_gain_folds  = []
    feat_imp_split_folds = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr  = X.iloc[tr_idx].copy()
        X_val = X.iloc[val_idx].copy()
        y_tr  = y.iloc[tr_idx]
        y_val = y.iloc[val_idx]

        # ── PASO 1: Target Encoding con sklearn.TargetEncoder ────────────────
        # fit_transform en train usa cross-fitting interno (cv=5 por defecto)
        # para evitar leakage. transform en val usa los encodings del fit
        # completo sobre train → correcto, sin contaminación del fold de val.
        #
        # IMPORTANTE: NO usar fit(X_tr, y_tr).transform(X_tr) — eso sí leakea.
        # fit_transform != fit + transform en este encoder (ver docs de sklearn).
        if te_cols:
            te = TargetEncoder(
                smooth=te_smoothing,      # Laplace smoothing; 'auto' = empirical Bayes
                target_type='continuous', # una sola col de salida por feature (no multiclass)
                cv=5,                     # folds del cross-fitting interno
                random_state=_seed,
            )
            # fit_transform aplica cross-fitting internamente sobre X_tr/y_tr
            te_tr  = te.fit_transform(X_tr[te_cols], y_tr)
            # transform usa los encodings del fit completo sobre train
            te_val = te.transform(X_val[te_cols])

            te_col_names = [f'{col}_te' for col in te_cols]
            for i, col_name in enumerate(te_col_names):
                X_tr[col_name]  = te_tr[:, i]
                X_val[col_name] = te_val[:, i]

        # ── PASO 2: Aggregations por grupo (solo con datos de train) ─────────
        if agg_config:
            for group_col, aggs in agg_config.items():
                
                # Smoothing: dict por group_col o valor global para todos
                if isinstance(agg_smoothing, dict):
                    sm = agg_smoothing.get(group_col, 0)  # 0 si no está definido
                else:
                    sm = agg_smoothing
                    
                # ── Asegurar que group_col es unidimensional antes de agrupar ────
                # Merges sucesivos pueden duplicar columnas si group_col aparece
                # tanto como clave de agrupación como columna fuente en otro agg.
                if isinstance(X_tr.index, pd.MultiIndex) or X_tr[group_col].ndim != 1:
                    X_tr[group_col] = X_tr[group_col].iloc[:, 0]
                    X_val[group_col] = X_val[group_col].iloc[:, 0]
                
                # ── Eliminar columnas duplicadas que pudo introducir el merge ────
                X_tr = X_tr.loc[:, ~X_tr.columns.duplicated()]
                X_val = X_val.loc[:, ~X_val.columns.duplicated()]
                
                agg_kwargs = {
                    new_name: pd.NamedAgg(column=src_col, aggfunc=func)
                    for new_name, (src_col, func) in aggs.items()
                }
        
                # ── Cross-fitting sobre train (misma lógica que TargetEncoder) ────
                # Cada fila de train se encodea con estadísticas calculadas
                # sin ella misma, usando los otros subfolds.
                n_cv = 5
                inner_skf = KFold(n_splits=n_cv, shuffle=True, random_state=_seed)
                new_col_names = list(aggs.keys())
        
                # Inicializar columnas en X_tr con NaN
                for new_name in new_col_names:
                    X_tr[new_name] = np.nan
        
                for inner_tr_idx, inner_val_idx in inner_skf.split(X_tr):
                    inner_X_tr = X_tr.iloc[inner_tr_idx]
        
                    agg_df_inner = inner_X_tr.groupby(group_col).agg(**agg_kwargs)
                    agg_df_inner['__n__'] = inner_X_tr.groupby(group_col).size()
                    agg_df_inner = agg_df_inner.reset_index()
        
                    global_means_inner = {
                        new_name: agg_df_inner[new_name].mean()
                        for new_name in new_col_names
                    }
        
                    # Mergear solo sobre las filas del subfold de val interno
                    inner_val_rows = X_tr.iloc[inner_val_idx][[group_col]].merge(
                        agg_df_inner, on=group_col, how='left'
                    )
        
                    for new_name in new_col_names:
                        gm = global_means_inner[new_name]
                        n  = inner_val_rows['__n__']
                        vals = (
                            (inner_val_rows[new_name] * n + gm * sm)
                            / (n + sm)
                        ).fillna(gm)
                        X_tr.iloc[inner_val_idx, X_tr.columns.get_loc(new_name)] = vals.values
        
                # ── Val: fit sobre todo train, transform sobre val ────────────────
                # Igual que TargetEncoder.transform(): usa estadísticas del fold
                # completo de train, sin ver val en ningún momento.
                agg_df_full = X_tr.groupby(group_col).agg(**agg_kwargs)
                agg_df_full['__n__'] = X_tr.groupby(group_col).size()
                agg_df_full = agg_df_full.reset_index()
        
                global_means_full = {
                    new_name: agg_df_full[new_name].mean()
                    for new_name in new_col_names
                }
        
                X_val = X_val.merge(agg_df_full, on=group_col, how='left')
                for new_name in new_col_names:
                    gm = global_means_full[new_name]
                    n  = X_val['__n__']
                    X_val[new_name] = (
                        (X_val[new_name] * n + gm * sm)
                        / (n + sm)
                    ).fillna(gm)
                X_val = X_val.drop(columns=['__n__'])
        
            agg_group_cols = list(agg_config.keys())
            X_tr  = X_tr.drop(columns=agg_group_cols, errors='ignore')
            X_val = X_val.drop(columns=agg_group_cols, errors='ignore')

        # ── DROP de columnas auxiliares globales ──────────────────────────────
        # Columnas que viajan en X para ser usadas por agg_config o tfidf_col
        # pero que no son features del modelo (strings, IDs, etc.)
        if aux_cols:
            X_tr  = X_tr.drop(columns=aux_cols, errors='ignore')
            X_val = X_val.drop(columns=aux_cols, errors='ignore')

        # ── PASO 4: Inicializar acumuladores en el primer fold ────────────────
        # (no antes, porque el shape final de X depende de TE/agg/tfidf)
        if fold == 0:
            feature_names  = list(X_tr.columns)
            n_feat         = len(feature_names)

        # ── PASO 4.5: Restaurar dtypes categóricos ───────────────────────────
        # Los merges y transformaciones pueden cambiar el dtype de columnas
        # categóricas (ej: int → float por NaNs introducidos en el join).
        # LightGBM requiere que las columnas declaradas en categorical_feature
        # sean int o category, nunca float u object.
        if cat_features:
            for col in cat_features:
                for df_ in [X_tr, X_val]:
                    if col in df_.columns:
                        df_[col] = df_[col].fillna(-1).astype(int)

        # ── PASO 5: Entrenamiento ─────────────────────────────────────────────
        model = lgb.LGBMClassifier(**params, random_state=_seed)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            categorical_feature=cat_features or 'auto',
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(period=-1),
            ]
        )

        proba = model.predict_proba(X_val)        # ← agregar
        oof_proba[val_idx] = proba                # ← agregar
        preds = proba.argmax(axis=1)             # ← reemplaza model.predict(X_val)
        oof_preds[val_idx] = preds
        fold_qwk.append(qwk(y_val, preds))
        fold_acc.append(accuracy_score(y_val, preds))
        fold_f1.append(f1_score(y_val, preds, average='macro'))
        feat_imp_gain_folds.append(model.booster_.feature_importance(importance_type='gain'))
        feat_imp_split_folds.append(model.booster_.feature_importance(importance_type='split'))

        if verbose:
            print(f'  Fold {fold+1}: QWK={fold_qwk[-1]:.4f}  '
                  f'Acc={fold_acc[-1]:.4f}  F1={fold_f1[-1]:.4f}  '
                  f'(best iter={model.best_iteration_})')

    gain_arr  = np.array(feat_imp_gain_folds)   # shape (n_folds, n_features)
    split_arr = np.array(feat_imp_split_folds)

    feat_imp_df = pd.DataFrame({
        'feature'  : feature_names,
        'gain'     : gain_arr.mean(axis=0),
        'gain_std' : gain_arr.std(axis=0),
        'split'    : split_arr.mean(axis=0),
        'split_std': split_arr.std(axis=0),
    }).sort_values('gain', ascending=False).reset_index(drop=True)

    results = {
        'name'     : experiment_name,
        'qwk_mean' : np.mean(fold_qwk),
        'qwk_std'  : np.std(fold_qwk),
        'acc_mean' : np.mean(fold_acc),
        'f1_mean'  : np.mean(fold_f1),
        'oof_preds': oof_preds,
        'oof_proba': oof_proba,
        'feat_imp' : feat_imp_df,
        'fold_qwk' : fold_qwk,
    }

    if verbose:
        print(f'\n  ► {experiment_name}')
        print(f'    QWK  {results["qwk_mean"]:.4f} ± {results["qwk_std"]:.4f}')
        print(f'    Acc  {results["acc_mean"]:.4f}')
        print(f'    F1   {results["f1_mean"]:.4f}')

    return results

class ExperimentTracker:
    """Registra y compara todos los experimentos del notebook."""
    def __init__(self):
        self.records = []

    def log(self, result, name=None):
        """
        result puede ser:
          - un dict de run_cv (experimento single-seed)
          - una lista de dicts (experimento multi-seed)
        name: nombre override (útil para multi-seed, ej: '00a_baseline_5seeds')
        """
        if isinstance(result, list):
            all_qwks = np.concatenate([r['fold_qwk'] for r in result])
            acc_mean = np.mean([r['acc_mean'] for r in result])
            f1_mean  = np.mean([r['f1_mean']  for r in result])
            exp_name = name or result[0]['name'].rsplit('_seed', 1)[0]  # strip seed suffix
            n_seeds  = len(result)
        else:
            all_qwks = np.array(result['fold_qwk'])
            acc_mean = result['acc_mean']
            f1_mean  = result['f1_mean']
            exp_name = name or result['name']
            n_seeds  = 1

        self.records.append({
            'Experimento': exp_name,
            'QWK'        : round(all_qwks.mean(), 4),
            'QWK±std'    : round(all_qwks.std(),  4),
            'Accuracy'   : round(acc_mean,         4),
            'F1 macro'   : round(f1_mean,          4),
            'n_seeds'    : n_seeds,
            '_fold_qwks' : all_qwks,   # guardado para delta() y gráficos futuros
        })

    def table(self):
        df_t = pd.DataFrame(self.records).drop(columns=['_fold_qwks'])
        return df_t.sort_values('QWK', ascending=False).reset_index(drop=True)

    def plot(self):
        df_t = pd.DataFrame(self.records).sort_values('QWK', ascending=False).reset_index(drop=True)
        fig, ax = plt.subplots(figsize=(10, 0.55 * len(df_t) + 2))
        colors_bar = [PALETTE[0] if i == 0 else PALETTE[2] for i in range(len(df_t))]

        bars = ax.barh(
            df_t['Experimento'][::-1],
            df_t['QWK'][::-1],
            xerr=df_t['QWK±std'][::-1],
            color=colors_bar[::-1],
            edgecolor='white',
            error_kw=dict(ecolor='grey', capsize=3, linewidth=1),
        )
        ax.bar_label(bars, fmt='%.4f', padding=6, fontsize=9)

        # Indicador de n_seeds en el eje Y
        ylabels = [
            f"{row['Experimento']} ({row['n_seeds']}s)"
            if row['n_seeds'] > 1 else row['Experimento']
            for _, row in df_t[::-1].iterrows()
        ]
        ax.set_yticklabels(ylabels)

        ax.set_title('Comparación de experimentos — QWK (CV)', fontweight='bold')
        ax.set_xlabel('QWK medio')
        ax.set_xlim(0, df_t['QWK'].max() + 0.08)
        plt.tight_layout()
        plt.show()


tracker = ExperimentTracker()
print('Infraestructura lista.')

In [ ]:
def plot_results(y_true, oof_preds, title=''):
    """
    oof_preds puede ser:
      - un array 1D (un solo experimento, como antes)
      - una lista de arrays (múltiples seeds → agrega con media ± std)
    """
    y_true = np.asarray(y_true).astype(int)
    labels  = ['0\nMismo día', '1\n1-7d', '2\n8-30d', '3\n31-90d', '4\nNo adopt.']
    classes = [0, 1, 2, 3, 4]

    # ── Normalizar input ──────────────────────────────────────────────────────
    if isinstance(oof_preds, list) and isinstance(oof_preds[0], np.ndarray):
        all_preds = [np.asarray(p).astype(int) for p in oof_preds]
        # Para la confusión usamos las predicciones promedio (voto por mayoría)
        preds_majority = np.stack(all_preds).T  # (n_samples, n_seeds)
        preds = np.apply_along_axis(
            lambda row: np.bincount(row, minlength=5).argmax(), 1, preds_majority
        )
        # Conteos por clase para cada seed → media y std
        p_counts = np.array([[(p == c).sum() for c in classes] for p in all_preds])
        p_mean   = p_counts.mean(axis=0)
        p_std    = p_counts.std(axis=0)
        multi_seed = True
    else:
        preds      = np.asarray(oof_preds).astype(int)
        p_mean     = np.array([(preds == c).sum() for c in classes], dtype=float)
        p_std      = None
        multi_seed = False

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # ── Matriz de confusión ───────────────────────────────────────────────────
    cm = confusion_matrix(y_true, preds, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues', ax=axes[0],
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, cbar_kws={'shrink': 0.8})
    axes[0].set_title(
        f'Matriz de confusión (recall por fila)\n'
        f'QWK={qwk(y_true, preds):.4f}  Acc={accuracy_score(y_true, preds):.4f}'
    )
    axes[0].set_ylabel('Real')
    axes[0].set_xlabel('Predicho')

    # ── Distribución real vs. predicha ────────────────────────────────────────
    x = np.arange(len(classes))
    w = 0.35
    r_counts = np.array([(y_true == c).sum() for c in classes], dtype=float)

    axes[1].bar(x - w/2, r_counts, w, label='Real',     color=PALETTE[0], edgecolor='white')
    axes[1].bar(x + w/2, p_mean,   w, label='Predicho', color=PALETTE[2], edgecolor='white',
                yerr=p_std,
                error_kw=dict(ecolor='grey', capsize=4, linewidth=1.2) if multi_seed else {})
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([str(c) for c in classes])
    axes[1].set_title(
        'Distribución real vs. predicha' +
        (f'\n(media ± std sobre {len(all_preds)} seeds)' if multi_seed else '')
    )
    axes[1].set_ylabel('Cantidad')
    axes[1].legend()

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(classification_report(
        y_true, preds,
        target_names=['0-MismoDía', '1-1ªSemana', '2-1erMes', '3-2do3erMes', '4-NoAdopt.']
    ))


def plot_importance(feat_imp_df, top_n=20, title='Feature Importance'):
    top              = feat_imp_df.sort_values('gain', ascending=False).head(top_n)
    features_ordered = top['feature'].values[::-1]

    fig, axes = plt.subplots(1, 2, figsize=(14, top_n * 0.3 + 1), sharey=True)

    axes[0].barh(
        features_ordered, top['gain'].values[::-1],
        xerr=top['gain_std'].values[::-1],
        color=PALETTE[0], edgecolor='white',
        error_kw=dict(ecolor='grey', capsize=3, linewidth=1),
    )
    axes[0].set_title('Gain', fontweight='bold')
    axes[0].set_xlabel('Importancia media (gain)')

    axes[1].barh(
        features_ordered, top['split'].values[::-1],
        xerr=top['split_std'].values[::-1],
        color=PALETTE[2], edgecolor='white',
        error_kw=dict(ecolor='grey', capsize=3, linewidth=1),
    )
    axes[1].set_title('Split', fontweight='bold')
    axes[1].set_xlabel('Importancia media (split)')

    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


def delta(result_new, result_ref):
    """
    Compara QWK entre dos experimentos.
    Cada argumento puede ser:
      - un dict resultado de run_cv (un solo experimento)
      - una lista de dicts (múltiples seeds → agrega automáticamente)
    """
    def extract_qwks(result):
        """Extrae todos los QWK por fold, de uno o varios resultados."""
        if isinstance(result, list):
            # Múltiples seeds: concatenar los fold_qwk de cada seed
            return np.concatenate([r['fold_qwk'] for r in result])
        else:
            return np.array(result['fold_qwk'])

    qwks_new = extract_qwks(result_new)
    qwks_ref = extract_qwks(result_ref)

    mean_new, std_new = qwks_new.mean(), qwks_new.std()
    mean_ref, std_ref = qwks_ref.mean(), qwks_ref.std()

    d    = mean_new - mean_ref
    sign = '+' if d >= 0 else ''

    print(f'  Referencia : QWK = {mean_ref:.4f} ± {std_ref:.4f}  (n={len(qwks_ref)} folds)')
    print(f'  Nuevo      : QWK = {mean_new:.4f} ± {std_new:.4f}  (n={len(qwks_new)} folds)')
    print(f'  ΔQWK       : {sign}{d:.4f}')

    return d


print('Funciones de visualización listas.')

---
# 3. Baseline — LightGBM sin Feature Engineering

Establece el piso de performance con features originales crudas.  
Cualquier experimento que no supere este número descarta sus features. Comenzaremos con todas las variables numéricas, y las variables categóricas que no tengan una cardinalidad alta (RescuerID, Name, Description).

In [ ]:
BASELINE_FEATURES = [
    # Numéricas originales
    'Age', 'Quantity', 'Fee', 'VideoAmt', 'PhotoAmt',
    # Categóricas ordinales (como enteros)
    'Type', 'Gender', 'Color1', 'Color2', 'Color3',
    'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized',
    'Health', 'State', 'Breed1', 'Breed2',
]

BASELINE_PARAMS = {
    'objective'        : 'multiclass',
    'num_class'        : 5,
    'metric'           : 'multi_logloss',
    'n_estimators'     : 2000,
    'learning_rate'    : 0.05,
    'num_leaves'       : 63,
    'min_child_samples': 20,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.1,
    'verbose'          : -1,
}

X_base = df[BASELINE_FEATURES].copy()
y      = df[TARGET].copy()

print(f'Shape baseline: {X_base.shape}')

In [ ]:
exp00a_baseline = []

for seed in SEEDS:
    # Sobreescribir la semilla global temporalmente
    #params = {**BASELINE_PARAMS, 'random_state': seed}
    
    r = run_cv(
        X=X_base,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        #cat_features=categoricals,
        experiment_name=f'00a_baseline_seed{seed}',
        verbose=False,  # para no saturar la salida
    )
    exp00a_baseline.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp00a_baseline, name='exp00a_baseline')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp00a_baseline]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp00a_baseline], title='Baseline — 5 seeds')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp00a_baseline])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Baseline — Top 25 features (5 seeds)')


## 3b Baseline con definición de categóricas
Se evaluará como definición preliminar si utilizar el parámetro de cat_features de LGBM mejora la predicción.

In [ ]:
categoricals = ['Type', 'Gender', 'Color1', 'Color2', 'Color3',
    'MaturitySize', 'FurLength', 'Vaccinated', 'Dewormed', 'Sterilized',
    'Health', 'State', 'Breed1', 'Breed2']

exp00b_baseline = []

for seed in SEEDS:
    # Sobreescribir la semilla global temporalmente
    #params = {**BASELINE_PARAMS, 'random_state': seed}
    
    r = run_cv(
        X=X_base,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp00b_baseline{seed}',
        verbose=False,  # para no saturar la salida
    )
    exp00b_baseline.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp00b_baseline, name='exp00b_baseline')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp00b_baseline]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")

delta(exp00b_baseline, exp00a_baseline)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp00b_baseline], title='Baseline — 5 seeds')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp00b_baseline])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Baseline — Top 25 features (5 seeds)')


Con una mejora leve en todas las métricas, y una disminución en el desvío estándar de los folds de CV, se observa que definir las variables categóricas según el parámetro de la librería de LGBM mejora la performance.
Se observa 

---
# 4. Feature Engineering iterativo

### Flujo de trabajo para agregar un experimento nuevo

1. Calcula las features **seguras** sobre todo el dataset `df_fe`
2. Define `CURRENT_FEATURES_XX = CURRENT_FEATURES + [nuevas_features]`
3. Llama `run_cv(X, y, params, ...)` con los parámetros que correspondan
4. Si mejora → actualiza `CURRENT_FEATURES = CURRENT_FEATURES_XX`

In [ ]:
# ── Estado acumulado ──────────────────────────────────────────────────────────
# CURRENT_FEATURES es la lista de features del mejor experimento hasta ahora.
# Cada bloque propone CURRENT_FEATURES_XX y lo incorpora solo si mejora el QWK.
CURRENT_FEATURES = BASELINE_FEATURES.copy()
df_fe = df.copy()   # DataFrame de trabajo; acumula todas las features nuevas
best_result = exp00b_baseline  # resultado del mejor experimento hasta ahora

print('FE iterativo listo.')

## 4.1a Variables de Name y Description

Extracción de señales simples del texto:
- Si tiene o no Nombre y Descripción.
- La cantidad de caracteres del Nombre y la Descripción.
- La cantidad de palabras de la Descripción.
- Un ratio entre cantidad de caracteres y de palabras, con la expectativa de que palabras más largas implican descripciones más complejas/elaboradas.

In [ ]:
# ── Señales de Name ───────────────────────────────────────────────────────────
df_fe['HasName']    = df_fe['Name'].notna().astype(int)
df_fe['NameLength'] = df_fe['Name'].str.len().fillna(0)

# ── Señales de Description ───────────────────────────────────────────────────
df_fe['HasDesc']       = df_fe['Description'].notna().astype(int)
df_fe['DescLength']    = df_fe['Description'].str.len().fillna(0)
df_fe['DescWordCount'] = df_fe['Description'].fillna('').str.split().str.len()
# Largo promedio de palabra: proxy de vocabulario usado
df_fe['DescRatio']     = df_fe['DescLength'] / df_fe['DescWordCount'].replace(0, np.nan)

NEW_VAR = ['HasName', 
          'NameLength', 
          'HasDesc', 
          'DescLength', 
          'DescWordCount', 
          'DescRatio']

CURRENT_FEATURES_41a = CURRENT_FEATURES + NEW_VAR
X_41a = df_fe[CURRENT_FEATURES_41a]

print(f'Features nuevas (4.1a): {NEW_VAR}')
print(f'Total features: {len(CURRENT_FEATURES_41a)}')
print('\n' + '='*55)
print('EXPERIMENTO 1a: + Señales de Name y Description')
print('='*55)

exp01_name_desc = []

for seed in SEEDS:
    # Sobreescribir la semilla global temporalmente
    #params = {**BASELINE_PARAMS, 'random_state': seed}
    
    r = run_cv(
        X=X_41a,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp01_name_desc{seed}',
        verbose=False,  # para no saturar la salida
    )
    exp01_name_desc.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp01_name_desc, name='exp01_name_desc')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp01_name_desc]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp01_name_desc, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp01_name_desc], title='exp01_name_desc')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp01_name_desc])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp01_name_desc')


El QWK cae muy levemente, y no presenta un desvío estándar que refleje mucha inestabilidad. Además, hay tres variables que alcanzan el top 5 de Feature Importance según Ganancia y Split, todas derivadas de la Descripción.
Por el contrario, las variables que identifican nombres o descripciones vacíos no aportan mucho valor. Se realizará un segundo experimento quitando las variables de HasName y HasDesc.

In [ ]:
NEW_VAR = [#'HasName', 
          'NameLength', 
          #'HasDesc', 
          'DescLength', 
          'DescWordCount', 
          'DescRatio']

CURRENT_FEATURES_41adesc = CURRENT_FEATURES + NEW_VAR
X_41adesc = df_fe[CURRENT_FEATURES_41adesc]

print(f'Features nuevas (4.1adesc): {NEW_VAR}')
print(f'Total features: {len(CURRENT_FEATURES_41adesc)}')
print('\n' + '='*55)
print('EXPERIMENTO 1a: + Señales de Description')
print('='*55)

exp01a_desc = []

for seed in SEEDS:
    # Sobreescribir la semilla global temporalmente
    #params = {**BASELINE_PARAMS, 'random_state': seed}
    
    r = run_cv(
        X=X_41adesc,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp01a_desc{seed}',
        verbose=False,  # para no saturar la salida
    )
    exp01a_desc.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp01a_desc, name='exp01a_desc')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp01a_desc]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp01a_desc, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp01a_desc], title='exp01a_desc')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp01a_desc])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp01a_desc')


El QWK disminuye suavemente. Se incorporarán todas las variables derivadas de Nombre y Descripción.

In [ ]:

CURRENT_FEATURES = CURRENT_FEATURES_41a
best_result = exp01_name_desc

## 4.1b Variables sobre fotos y videos.


Transformación de las variables de PhotoAmt y VideoAmt:
- Si tiene Foto o Video.
- Una transformación logarítmica de las variables de PhotoAmt y VideoAmt, que presentan asimetría.

In [ ]:
print(f'Asimetría de PhotoAmt: {df_fe["PhotoAmt"].skew():.4f}')
print(f'Asimetría de VideoAmt: {df_fe["VideoAmt"].skew():.4f}')

In [ ]:
df_fe['HasPhoto']   = (df_fe['PhotoAmt'].fillna(0) > 0).astype(int)
df_fe['HasVideo']   = (df_fe['VideoAmt'].fillna(0) > 0).astype(int)
df_fe['PhotoAmt_log'] = np.log1p(df_fe['PhotoAmt'])
df_fe['VideoAmt_log'] = np.log1p(df_fe['VideoAmt'])

NEW_VAR = ['HasPhoto', 
          'HasVideo', 
          'PhotoAmt_log',
          'VideoAmt_log'
         ]

features_sin_originales = [f for f in CURRENT_FEATURES 
                            if f not in ['PhotoAmt', 
                                         'VideoAmt'
                                         ]]

CURRENT_FEATURES_41b_reemplazo = features_sin_originales + NEW_VAR
X_41brep = df_fe[CURRENT_FEATURES_41b_reemplazo]

print(f'Features nuevas (4.1b): {NEW_VAR}')
print(f'Total features: {len(CURRENT_FEATURES_41b_reemplazo)}')
print('\n' + '='*55)
print('EXPERIMENTO 1b: + Videos y fotos con reemplazo')
print('='*55)


exp01b_photo_vid = []

for seed in SEEDS:
    # Sobreescribir la semilla global temporalmente
    #params = {**BASELINE_PARAMS, 'random_state': seed}
    
    r = run_cv(
        X=X_41brep,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp01b_photo_vid{seed}',
        verbose=False,  # para no saturar la salida
    )
    exp01b_photo_vid.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp01b_photo_vid, name='exp01b_photo_vid')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp01b_photo_vid]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp01b_photo_vid, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp01b_photo_vid], title='exp01b_photo_vid')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp01b_photo_vid])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp01b_photo_vid')


Las transformaciones sobre estas variables no generan mejoras. No se incorporarán.

## 4.1c Variables derivadas del Estado

La variable de Estado es una categórica con cardinalidad relativamente alta. Se incorpora información externa sobre esta variable para aportar contexto:
- Población en miles de cada uno de los Estados, según información del Departamento de Estadísticas de Malasia.[https://open.dosm.gov.my/data-catalogue/population_malaysia]
- Ingreso mediano de cada uno de los Estados, según el Departamento de Estadísticas de Malasia.[https://open.dosm.gov.my/data-catalogue/hh_income_state]
- Matrimonios absolutos por año y Rate de matrimonio por año de cada uno de los Estados. [https://open.dosm.gov.my/data-catalogue/marriages_state]

In [ ]:
malaysia_pop = pd.read_parquet('https://storage.dosm.gov.my/population/population_state.parquet')
#malaysia_pop = pd.read_parquet('/kaggle/input/datasets/tarmencello/state-metrics/population_state.parquet')
malaysia_pop = malaysia_pop[(malaysia_pop['ethnicity'] == 'overall') & 
                            (malaysia_pop['age'] == 'overall') & 
                            (malaysia_pop['sex'] == 'both') & 
                            (malaysia_pop['date'] == datetime.date(2022, 1, 1))][['state', 'population']]

malaysia_inc = pd.read_parquet('https://storage.dosm.gov.my/hies/hh_income_state.parquet')
#malaysia_inc = pd.read_parquet('/kaggle/input/datasets/tarmencello/state-metrics/hh_income_state.parquet')
malaysia_inc = malaysia_inc[malaysia_inc['date'] == datetime.date(2022, 1, 1)][['state', 'income_median']]

malaysia_mar = pd.read_parquet('https://storage.dosm.gov.my/demography/marriages_state.parquet')
#malaysia_mar = pd.read_parquet('/kaggle/input/datasets/tarmencello/state-metrics/marriages_state.parquet')
malaysia_mar = malaysia_mar[malaysia_mar['date'] == datetime.date(2022, 1, 1)]
malaysia_mar = malaysia_mar[['state', 'rate']].rename(columns={'rate': 'marriage_rate'})
malaysia_mar = malaysia_mar.groupby('state').sum()

El dataset del DOSM separa el Territorio Federal de Putrajaya para reportar las estadísticas. Sin embargo el dataset de etiquetas de PetFinder no lo incluye entre los Estados registrados. Por eso se incorporará al Estado de Selangor (usualmente se lo considera como parte debido a proximidad geográfica), recalculando las métricas con promedios ponderados por población.

In [ ]:
malaysia_stats = malaysia_pop.merge(malaysia_inc, on='state').merge(malaysia_mar, on='state')
malaysia_stats = malaysia_stats.rename(columns = {'state' : 'StateName'})

mapping = {
    'W.P. Kuala Lumpur': 'Kuala Lumpur',
    'W.P. Labuan': 'Labuan',
    'W.P. Putrajaya': 'Selangor'
}

malaysia_stats['StateName'] = malaysia_stats['StateName'].replace(mapping)

malaysia_stats['w_income_median'] = malaysia_stats['income_median'] * malaysia_stats['population']
malaysia_stats['w_marriage_rate'] = malaysia_stats['marriage_rate'] * malaysia_stats['population']

agg_stats = malaysia_stats.groupby('StateName').agg({
    'population': 'sum',
    'w_income_median': 'sum',
    'w_marriage_rate': 'sum'
}).reset_index()

agg_stats['income_median'] = agg_stats['w_income_median'] / agg_stats['population']
agg_stats['marriage_rate'] = agg_stats['w_marriage_rate'] / agg_stats['population']

malaysia_stats = agg_stats[['StateName', 'population', 'income_median', 'marriage_rate']]

malaysia_stats = states.merge(malaysia_stats, on='StateName', how= 'outer')
print(malaysia_stats.sort_values('population', ascending=False))


In [ ]:

df_fe['StatePop']       = df_fe['State'].map(malaysia_stats.set_index('StateID')['population'].to_dict())
df_fe['StateIncomeMedian'] = df_fe['State'].map(malaysia_stats.set_index('StateID')['income_median'].to_dict())
df_fe['StateMarriageRate'] = df_fe['State'].map(malaysia_stats.set_index('StateID')['marriage_rate'].to_dict())


NEW_VAR = ['StatePop', 
          'StateIncomeMedian',
          'StateMarriageRate'
         ]

CURRENT_FEATURES_41c = CURRENT_FEATURES + NEW_VAR
X_41c = df_fe[CURRENT_FEATURES_41c]

print(f'Features nuevas (4.1c): {NEW_VAR}')
print(f'Total features: {len(CURRENT_FEATURES_41c)}')
print('\n' + '='*55)
print('EXPERIMENTO 1c: + Contexto de Estados')
print('='*55)


exp01c_states = []

for seed in SEEDS:
    r = run_cv(
        X=X_41c,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp01c_states{seed}',
        verbose=False,  
    )
    exp01c_states.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp01c_states, name='exp01c_states')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp01c_states]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp01c_states, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp01c_states], title='exp01c_states')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp01c_states])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp01c_states')


Las variables que aportan contexto sobre los Estados en que se adoptan los animales parecen aportar un ligero valor. Particularmente la variable que reporta la Población. Además, parecen mejorar la señal que aportaba la variable de State. Se replicará el experimento eliminando la variable de origen para identificar si la performance es mejor, o si las variables son complementarias.

In [ ]:
features_sin_originales = [f for f in CURRENT_FEATURES
                            if f not in ['State']]

categoricals_sin_originales = [f for f in categoricals 
                            if f not in ['State']]


CURRENT_FEATURES_41c_reemplazo = features_sin_originales + NEW_VAR
X_41c_reemplazo = df_fe[CURRENT_FEATURES_41c_reemplazo]

print(f'Features nuevas (4.1c rep): {NEW_VAR}')
print(f'Total features: {len(CURRENT_FEATURES_41c_reemplazo)}')
print('\n' + '='*55)
print('EXPERIMENTO 1c: + Contexto de Estados con reemplazo')
print('='*55)

exp01c_states_reemplazo = []

for seed in SEEDS:
    r = run_cv(
        X=X_41c_reemplazo,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals_sin_originales,
        experiment_name=f'exp01c_states{seed}',
        verbose=False,  
    )
    exp01c_states_reemplazo.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp01c_states_reemplazo, name='exp01c_states_reemplazo')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp01c_states_reemplazo]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp01c_states_reemplazo, best_result)


El aporte parece ser complementario. Se mantiene la variable original y se incorporan las contextuales.

In [ ]:
CURRENT_FEATURES = CURRENT_FEATURES_41c
best_result = exp01c_states

## 4.1d Variables de características del animal

Se realizan transformaciones para generar:
- Flags de características como si es de 1 sólo color, de raza pura
- Si la publicación corresponde a un grupo de animales o a uno solo.

In [ ]:
df_fe['IsFree']     = (df_fe['Fee'] == 0).astype(int)
df_fe['MixedBreed'] = (
    (df_fe['Breed2Name'].notna() & (df_fe['Breed1'] != df_fe['Breed2'])) |
    (df_fe['Breed1Name'] == 'Mixed Breed') |
    (df_fe['Breed1Name'].str.contains('Domestic', na=False))
).astype(int)
df_fe['MixedColor'] = ((df_fe['Color2'] != 0) & df_fe['Color2'].notna()).astype(int)
df_fe['IsGroup']    = (df_fe['Quantity'] > 1).astype(int)

NEW_VAR = ['IsFree', 
          'MixedBreed',
          'MixedColor',
          'IsGroup'
         ]

CURRENT_FEATURES_41d = CURRENT_FEATURES + NEW_VAR
X_41d = df_fe[CURRENT_FEATURES_41d]

print(f'Features nuevas (4.1d): {NEW_VAR}')
print(f'Total features: {len(CURRENT_FEATURES_41d)}')
print('\n' + '='*55)
print('EXPERIMENTO 1d: + Características del animal')
print('='*55)


exp01d_animal = []

for seed in SEEDS:
    r = run_cv(
        X=X_41d,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp01d_animal{seed}',
        verbose=False,  
    )
    exp01d_animal.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp01d_animal, name='exp01d_animal')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp01d_animal]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp01d_animal, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp01d_animal], title='exp01d_animal')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp01d_animal])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp01d_animal')


Estas variables no mejoran la performance del modelo. Se analizará la performance al eliminar las variables de origen Breed2, Color2 y Color3.

In [ ]:
NEW_VAR = ['MixedBreed',
          'MixedColor'
         ]

features_sin_originales = [f for f in CURRENT_FEATURES
                            if f not in ['Breed2', 'Color2', 'Color3']]

categoricals_sin_originales = [f for f in categoricals 
                            if f not in ['Breed2', 'Color2', 'Color3']]

CURRENT_FEATURES_41d_reemplazo = features_sin_originales + NEW_VAR
X_41d_reemplazo = df_fe[CURRENT_FEATURES_41d_reemplazo]

print(f'Features nuevas (4.1d): {NEW_VAR}')
print(f'Total features: {len(CURRENT_FEATURES_41d_reemplazo)}')
print('\n' + '='*55)
print('EXPERIMENTO 1d: + Características del animal')
print('='*55)

exp01d_animal_rep = []

for seed in SEEDS:
    r = run_cv(
        X=X_41d_reemplazo,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals_sin_originales,
        experiment_name=f'exp01d_animal{seed}',
        verbose=False,  
    )
    exp01d_animal_rep.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp01d_animal_rep, name='exp01d_animal_rep')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp01d_animal_rep]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp01d_animal_rep, exp01d_animal)

## 4.1e Variables relativas a la salud del animal
Las transformaciones buscan:
- Dicotomizar la información de si el animal fue Vacunado o no, Desparasitado o no, y Esterilizado o no (donde Sí = 1 y No / No sabe = 0)
- Generar un Score de salud general, identificando las características del animal.

In [ ]:
# ── Score de salud combinado ──────────────────────────────────────────────────
# Vaccinated/Dewormed/Sterilized: 1=Sí, 2=No, 3=No sabe
# Health: 1=Sano, 2=Lesión leve, 3=Lesión grave, 0=No especificado

df_fe['VaccYes']   = (df_fe['Vaccinated']  == 1).astype(int)
df_fe['DewormYes'] = (df_fe['Dewormed']    == 1).astype(int)
df_fe['SterYes']   = (df_fe['Sterilized']  == 1).astype(int)
df_fe['Healthy']   = (df_fe['Health']  == 1).astype(int)

health_map = {0: np.nan, 1: 1.0, 2: 0.5, 3: 0.0}
df_fe['HealthScore'] = ((df_fe['VaccYes'] + df_fe['DewormYes'] + df_fe['SterYes'])/3 * 0.4 ) + (df_fe['Health'].replace(health_map) * 0.6)

NEW_VAR = ['VaccYes', 
          'DewormYes',
          'SterYes',
          'HealthScore',
          'Healthy'
         ]

CURRENT_FEATURES_41e = CURRENT_FEATURES + NEW_VAR
X_41e = df_fe[CURRENT_FEATURES_41e]

print(f'Features nuevas (4.1e): {NEW_VAR}')
print(f'Total features: {len(CURRENT_FEATURES_41e)}')
print('\n' + '='*55)
print('EXPERIMENTO 1e: + Salud del animal')
print('='*55)


exp01e_health = []

for seed in SEEDS:
    r = run_cv(
        X=X_41e,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp01e_health{seed}',
        verbose=False,  
    )
    exp01e_health.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp01e_health, name='exp01e_health')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp01e_health]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp01e_health, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp01e_health], title='exp01e_health')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp01e_health])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp01e_health')


Se observa un efecto al incorporar estas nuevas features. Se evalúa el mismo experimento pero eliminando las variables origen del modelo.

In [ ]:
NEW_VAR = ['VaccYes', 
          'DewormYes',
          'SterYes',
          'HealthScore',
          'Healthy'
         ]

features_sin_originales = [f for f in CURRENT_FEATURES 
                            if f not in ['Vaccinated', 
                                         'Dewormed', 
                                         'Sterilized', 
                                         'Health']]

categoricals_sin_originales = [f for f in categoricals 
                            if f not in ['Vaccinated', 'Dewormed', 'Sterilized', 'Health']]


CURRENT_FEATURES_41e_reemplazo = features_sin_originales + NEW_VAR
X_41erep = df_fe[CURRENT_FEATURES_41e_reemplazo]

print(f'Features nuevas (4.1e): {NEW_VAR}')
print(f'Total features: {len(CURRENT_FEATURES_41e_reemplazo)}')
print('\n' + '='*55)
print('EXPERIMENTO 1e: + Salud del animal con reemplazo')
print('='*55)

exp01e_health_rep = []

for seed in SEEDS:
    r = run_cv(
        X=X_41erep,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals_sin_originales,
        experiment_name=f'exp01e_health_rep{seed}',
        verbose=False,  
    )
    exp01e_health_rep.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp01e_health_rep, name='exp01e_health_rep')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp01e_health_rep]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp01e_health, best_result)

Estas variables no parecen aportar valor.

## 4.2 Transformaciones de variables numéricas

Se aplicarán logaritmos naturales a las variables numéricas Age, Fee y Quantity. Esto busca reducir la asimetría de estas variables para evaluar si mejora la relación con la variable objetivo.

In [ ]:
print(f"Asimetría de Age: {stats.skew(df_fe['Age'])} - Positiva")
print(f"Asimetría de Fee: {stats.skew(df_fe['Fee'])} - Positiva")
print(f"Asimetría de Quantity: {stats.skew(df_fe['Quantity'])} - Positiva")


In [ ]:
# ── Transformaciones de skewness ──────────────────────────────────────────────
df_fe['Age_log']      = np.log1p(df_fe['Age'])
df_fe['Fee_log']      = np.log1p(df_fe['Fee'])
df_fe['Qty_log']      = np.log1p(df_fe['Quantity'])

NEW_NUM = ['Age_log',
          'Fee_log',
          'Qty_log']

CURRENT_FEATURES_42 = CURRENT_FEATURES + NEW_NUM
X_42 = df_fe[CURRENT_FEATURES_42]

print(f'Features nuevas (4.2): {NEW_NUM}')
print(f'Total features: {len(CURRENT_FEATURES_42)}')
print('\n' + '='*55)
print('EXPERIMENTO 2: + Transformaciones numéricas')
print('='*55)

exp02_logs = []

for seed in SEEDS:
    r = run_cv(
        X=X_42,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp02_logs{seed}',
        verbose=False,  
    )
    exp02_logs.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp02_logs, name='exp02_logs')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp02_logs]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp02_logs, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp02_logs], title='exp02_logs')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp02_logs])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp02_logs')


Si bien no aporta mejoras en QWK, las métricas parecen tener valor en el Feature Importance. Se evalúa el mismo modelo pero eliminando las variables de origen.

In [ ]:
NEW_NUM = [#'Age_log',
          'Fee_log',
          #'Qty_log'
          ]

features_sin_originales = [f for f in CURRENT_FEATURES 
                            if f not in [#'Age', 
                                         'Fee', 
                                         #'Quantity'
                            ]]

CURRENT_FEATURES_42_reemplazo = features_sin_originales + NEW_NUM
X_42rep = df_fe[CURRENT_FEATURES_42_reemplazo]

print(f'Features nuevas (4.2): {NEW_NUM}')
print(f'Total features: {len(CURRENT_FEATURES_42_reemplazo)}')
print('\n' + '='*55)
print('EXPERIMENTO 2: + Logaritmos con reemplazo')
print('='*55)

exp02_logs_rep = []

for seed in SEEDS:
    r = run_cv(
        X=X_42rep,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp02_logs_rep{seed}',
        verbose=False,  
    )
    exp02_logs_rep.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp02_logs_rep, name='exp02_logs_rep')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp02_logs_rep]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp02_logs_rep, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp02_logs_rep], title='exp02_logs_rep')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp02_logs_rep])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp02_logs_rep')


La única variable que parece mejorar al convertirse a logaritmo natural es Fee. Se incorpora esta variable sin su original.

In [ ]:
CURRENT_FEATURES = CURRENT_FEATURES_42_reemplazo
best_result = exp02_logs_rep

## 4.3 Features de interacción

Multiplicaciones entre features. Se busca identificar interacciones entre variables que puedan ser explicativas de la variable objetivo, optimizando los cortes que realiza el algoritmo. 

In [ ]:
df_fe['Type_x_Age']    = df_fe['Type']    * df_fe['Age']
df_fe['Type_x_IsFree'] = df_fe['Type']    * df_fe['IsFree']
df_fe['Age_x_IsFree']  = df_fe['Age'] * df_fe['IsFree']
df_fe['Fee_x_Photo']   = df_fe['Fee'] * df_fe['PhotoAmt']
df_fe['Age_x_Photo']   = df_fe['Age'] * df_fe['PhotoAmt']
df_fe['Health_x_Type'] = df_fe['Health']  * df_fe['Type']
df_fe['Type_x_Mixed']  = df_fe['Type']    * df_fe['MixedBreed']

NEW_INT = ['Type_x_Age', 
           'Type_x_IsFree', 
           'Age_x_IsFree',
           'Fee_x_Photo', 
           'Age_x_Photo', 
           'Health_x_Type', 
           'Type_x_Mixed'
          ]

CURRENT_FEATURES_43 = CURRENT_FEATURES + NEW_INT
X_43 = df_fe[CURRENT_FEATURES_43]

print(f'Features nuevas (4.3): {NEW_INT}')
print(f'Total features: {len(CURRENT_FEATURES_43)}')
print('\n' + '='*55)
print('EXPERIMENTO 3: + Interacciones')
print('='*55)

exp03_interactions = []

for seed in SEEDS:
    r = run_cv(
        X=X_43,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'exp03_interactions{seed}',
        verbose=False,  
    )
    exp03_interactions.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp03_interactions, name='exp03_interactions')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp03_interactions]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp03_interactions, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp03_interactions], title='exp03_interactions')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp03_interactions])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp03_interactions')


Estas variables parecen generar interferencia. Particularmente se observa con la variable Age, que es utilizada en varios cruces y la variable original pierde mucho peso. Dado que el QWK aumentado es muy pequeño y que crece la variabilidad, no se incorporarán estas variables.

## 4.4a Aggregations por RescuerID
Estadísticas de comportamiento del rescatista: se genererará una columnaq ue identifique, dentro de cada fold del proceso de Cross-validation, si el rescatista es nuevo (es decir, si sólo hay una publicación de ese rescatista dentro del fold) o no. Como esta variable es dicotómica y no estadística, no se aplicará smoothing.

In [ ]:
# ── Definición de las aggregations ────────────────────────────────────────────
# Formato: {columna_de_agrupación: {nombre_nueva_col: (col_fuente, función)}}
AGG_RESCUER = {
    'RescuerID': {
        'resc_is_new' : ('RescuerID',  lambda x: int(len(x) == 1)),
    }
}

AGG_SMOOTHING = {
    'RescuerID': 0
}

# RescuerID debe estar en X para que run_cv pueda agrupar
AUX_COLS = ['RescuerID']
CURRENT_FEATURES_44a = CURRENT_FEATURES
X_44a = df_fe[CURRENT_FEATURES_44a + AUX_COLS]

print('Features que se generarán dentro del fold:')
print([k for v in AGG_RESCUER.values() for k in v.keys()])
print(f'\nTotal features en X (incl. RescuerID auxiliar): {len(CURRENT_FEATURES_44a)}')
print('\n' + '='*55)
print('EXPERIMENTO 4a: + Aggregations por rescatista')
print('='*55)

exp04a_aggregations = []

for seed in SEEDS:
    r = run_cv(
        X=X_44a,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        agg_config=AGG_RESCUER,
        agg_smoothing=AGG_SMOOTHING,       # rescatistas con <n publicaciones tiran a la media
        aux_cols=AUX_COLS, 
        experiment_name=f'exp04a_aggregations{seed}',
        verbose=False,  
    )
    exp04a_aggregations.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp04a_aggregations, name='exp04a_aggregations')

# Resumen final
all_qwks = [r['qwk_mean'] for r in exp04a_aggregations]
print(f"\n── Resumen sobre {len(SEEDS)} semillas ──")
print(f"QWK medio:  {np.mean(all_qwks):.4f}")
print(f"QWK std:    {np.std(all_qwks):.4f}")
print(f"QWK rango:  [{min(all_qwks):.4f}, {max(all_qwks):.4f}]")
delta(exp04a_aggregations, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp04a_aggregations], title='exp04a_aggregations')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp04a_aggregations])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp04a_aggregations')


Las variables agregadas a partir del Rescatista parecen cobrar mucha importancia. Esto puede deberse a que capturan información valiosa que correlaciona con la variable target, aunque también puede ser Leakage encubierto.
Para evaluar si este experimento genera Leakage, se realizará un experimento aleatorizando la variable target y evaluando el desempeño del modelo para identificar una memorización de falsos patrones.

In [ ]:
# ── Test de leakage para aggregations ────────────────────────────────────────
# Si hay leakage, un target aleatorio también va a "mejorar" con las aggregations
# porque las estadísticas del grupo van a correlacionar con el target shuffleado

y_shuffled = y.sample(frac=1, random_state=SEED).reset_index(drop=True)


exp04a_aggregations_shuffled = []

for seed in SEEDS:
    r = run_cv(
        X=X_44a,
        y=y_shuffled,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        agg_config=AGG_RESCUER,
        agg_smoothing=AGG_SMOOTHING,       # rescatistas con <n publicaciones tiran a la media
        aux_cols=AUX_COLS, 
        experiment_name=f'exp04a_aggregations_shuffled{seed}',
        verbose=False,  
    )
    exp04a_aggregations_shuffled.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp04a_aggregations_shuffled, name='exp04a_aggregations_shuffled')

baseline_shuffled = []
for seed in SEEDS:
    r = run_cv(
        X=X_base,
        y=y_shuffled,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        experiment_name=f'baseline_shuffled{seed}',
        verbose=False,  
    )
    baseline_shuffled.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(baseline_shuffled, name='baseline_shuffled')

# Resumen final
print(f'\nQWK baseline con target real:     {np.mean([r['qwk_mean'] for r in exp00b_baseline]):.4f} ± {np.std([r['qwk_mean'] for r in exp00b_baseline]):.4f}')
print(f'QWK baseline con target shuffle:  {np.mean([r['qwk_mean'] for r in baseline_shuffled]):.4f} ± {np.std([r['qwk_mean'] for r in baseline_shuffled]):.4f}')
print(f'QWK rescuer agg con target real:  {np.mean([r['qwk_mean'] for r in exp04a_aggregations]):.4f} ± {np.std([r['qwk_mean'] for r in exp04a_aggregations]):.4f}')
print(f'QWK rescuer agg con target shuffle: {np.mean([r['qwk_mean'] for r in exp04a_aggregations_shuffled]):.4f} ± {np.std([r['qwk_mean'] for r in exp04a_aggregations_shuffled]):.4f}')

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp04a_aggregations_shuffled], title='exp04a_aggregations_shuffled')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp04a_aggregations_shuffled])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp04a_aggregations_shuffled')


El QWK se desploma, y las variables de agregación de RescuerID no destacan frente a otras. Se incorporará la feature.

In [ ]:
CURRENT_FEATURES = CURRENT_FEATURES_44a
best_result = exp04a_aggregations

## 4.4b Aggregations por Breed

Estadísticas de comportamiento de la raza: edad promedio, frecuencia.

In [ ]:
# ── Distribución de frecuencias por RescuerID ─────────────────────────────────
freq_rescuer = df_fe['Breed1'].value_counts()

media   = freq_rescuer.mean()
mediana = freq_rescuer.median()

fig, ax = plt.subplots(figsize=(12, 4))

ax.hist(freq_rescuer.values, bins=50, color=PALETTE[0], edgecolor='white')

ax.axvline(media,   color=PALETTE[1], linewidth=2, linestyle='--', label=f'Media:   {media:.1f}')
ax.axvline(mediana, color=PALETTE[2], linewidth=2, linestyle='-',  label=f'Mediana: {mediana:.1f}')

ax.set_title('Distribución de publicaciones por Raza Principal', fontweight='bold')
ax.set_xlabel('Cantidad de publicaciones')
ax.set_ylabel('Cantidad de Razas')
ax.legend()

plt.tight_layout()
plt.show()

# ── Resumen estadístico ───────────────────────────────────────────────────────
print(f'Total razas únicas: {len(freq_rescuer):,}')
print(f'Media:                    {media:.1f}')
print(f'Mediana:                  {mediana:.1f}')
print(f'Máximo:                   {freq_rescuer.max()}')
print(f'% con 1 sola publicación: {(freq_rescuer == 1).mean()*100:.1f}%')
print(f'% con ≤2 publicaciones:   {(freq_rescuer <= 2).mean()*100:.1f}%')
print(f'% con ≤5 publicaciones:   {(freq_rescuer <= 5).mean()*100:.1f}%')
print(f'% con ≤10 publicaciones:  {(freq_rescuer <= 10).mean()*100:.1f}%')
print(f'% con ≤20 publicaciones:  {(freq_rescuer <= 20).mean()*100:.1f}%')
print(f'% con ≤50 publicaciones:  {(freq_rescuer <= 50).mean()*100:.1f}%')

La cardinalidad de las razas principales es menor que la de los rescatistas, y la distribución es algo más equilibrada. Se define un smoothing más pequeño.

In [ ]:
# ── Definición de las aggregations ────────────────────────────────────────────
# Formato: {columna_de_agrupación: {nombre_nueva_col: (col_fuente, función)}}
AGG_BREED = {
    'Breed1': {
        'breed_median_age'    : ('Age',      'median'),
        'breed_frecuency' : ('Breed1', 'count')
    }
}

AGG_E04b = {**AGG_RESCUER, **AGG_BREED}

AGG_SMOOTHING = {
    'RescuerID': 0,
    'Breed1':    25
}

# RescuerID debe estar en X para que run_cv pueda agrupar
AUX_COLS = ['RescuerID']
CURRENT_FEATURES_44b = CURRENT_FEATURES_43
X_44b = df_fe[CURRENT_FEATURES_44b + AUX_COLS]

print('Features que se generarán dentro del fold:')
print([k for v in AGG_E04b.values() for k in v.keys()])
print(f'\nTotal features en X (incl. RescuerID auxiliar): {len(CURRENT_FEATURES_44b)}')
print('\n' + '='*55)
print('EXPERIMENTO 4b: + Aggregations por raza')
print('='*55)

categoricals_4b = [f for f in categoricals if f not in ['Breed1']]

exp04b_breed_agg = []
for seed in SEEDS:
    r = run_cv(
        X=X_44b,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals_4b,
        agg_config=AGG_E04b,
        agg_smoothing=AGG_SMOOTHING,       # razas con <n publicaciones tiran a la media
        aux_cols=AUX_COLS,
        experiment_name=f'exp04b_breed_agg{seed}',
        verbose=False,  
    )
    exp04b_breed_agg.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp04b_breed_agg, name='exp04b_breed_agg')
delta(exp04b_breed_agg, best_result)


In [ ]:
plot_results(y, [r['oof_preds'] for r in exp04b_breed_agg], title='exp04b_breed_agg')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp04b_breed_agg])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp04b_breed_agg')


Las variables de raza no aportan más valor que las de RescuerID. Al excluir del modelo la variable cruda Breed1, el modelo pierde mucha fuerza que no se equipara con las variables incorporadas.
Por el momento no se incluirán.

## 4.5 Target Encoding

Variables de alta cardinalidad (`Breed1`) codificadas
con la media suavizada del target.
No se mezclará el experimento de Target Encoding con el de Agregaciones para evitar generar ruido o interferencia entre las variables.

**Anti-leakage:** el parámetro `te_cols` le indica a `run_cv` que calcule
el encoding dentro de cada fold usando solo `y_tr`. El suavizado
hace que categorías con pocas observaciones tiren hacia la media global,
reduciendo el riesgo de overfitting en categorías raras.

In [ ]:
TE_COLS = ['Breed1']
# run_cv generará automáticamente col+'_te' para cada una.

X_45 = df_fe[CURRENT_FEATURES_41e + AUX_COLS]   # usa el mejor conjunto hasta ahora

print(f'Columnas a encodear: {TE_COLS}')
print(f'Columnas generadas dentro del fold: {[c+"_te" for c in TE_COLS]}')
print('\n' + '='*55)
print('EXPERIMENTO 5: + Target Encoding (Breed1, RescuerID)')
print('='*55)

exp05_te = []
for seed in SEEDS:
    r = run_cv(
        X=X_45,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        agg_config=AGG_RESCUER,       # para comparar con las aggregations
        agg_smoothing=AGG_SMOOTHING,     
        te_cols=TE_COLS,       # ← calculado dentro del fold, sin leakage
        te_smoothing='auto',
        aux_cols=AUX_COLS, 
        experiment_name=f'exp05_te{seed}',
        verbose=False,  
    )
    exp05_te.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp05_te, name='exp05_te')
delta(exp05_te, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp05_te], title='exp05_te')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp05_te])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp05_te')


Al igual que con la agregación de Breed1, esta variable no logra equiparar el valor aportado por la original Breed1.
No se incorporará al modelo.

---
# 5 Análisis de texto

<!-- ## 5.1 Análisis de sentimiento a través de lexicon - NLTK
Se comenzará haciendo un análisis sencillo a través de diccionario, con la librería NLTK, para evaluar si se registran mejoras en el modelo. -->

## 5.1 Análisis de emociones a través de lexicon
Se utilizará la librería pysentimiento que permite hacer análisis de emociones.

In [ ]:
df_fe['DescriptionClean'] = df_fe['Description'].fillna('')
df_fe['DescriptionClean'] = df_fe['DescriptionClean'].str.strip()
df_fe['DescriptionClean'] = df_fe['DescriptionClean'].str[:512]

sia = SentimentIntensityAnalyzer()

def vader_sentiment(texto):
    if not texto:
        return None, None, None, None
    scores = sia.polarity_scores(texto)
    return scores['pos'], scores['neg'], scores['neu'], scores['compound']

(df_fe['vader_pos'],
 df_fe['vader_neg'],
 df_fe['vader_neu'],
 df_fe['vader_compound']) = zip(*df_fe['DescriptionClean'].apply(vader_sentiment))

# Etiqueta resumen basada en compound score
def vader_label(compound):
    if compound is None:
        return None
    if compound >= 0.05:
        return 'POS'
    elif compound <= -0.05:
        return 'NEG'
    else:
        return 'NEU'

df_fe['vader_label'] = df_fe['vader_compound'].apply(vader_label)

In [ ]:
label_map = {'POS': 1, 'NEU': 0, 'NEG': -1}
df_fe['vader_label_enc'] = df_fe['vader_label'].map(label_map)

NEW_INT = ['vader_label_enc']
CURRENT_FEATURES_51 = CURRENT_FEATURES_42_reemplazo + AUX_COLS + NEW_INT
X_51 = df_fe[CURRENT_FEATURES_51]

print(f'Features nuevas (5.1): {NEW_INT}')
print(f'Total features: {len(CURRENT_FEATURES_51)}')
print('\n' + '='*55)
print('EXPERIMENTO 5.1: + Sentiment Analysis (Lexicon)')
print('='*55)

exp051_sentiment_lexicon = []

for seed in SEEDS:
    r = run_cv(
        X=X_51,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals + NEW_INT,
        agg_config=AGG_RESCUER,
        agg_smoothing=AGG_SMOOTHING,       # rescatistas con <n publicaciones tiran a la media
        aux_cols=AUX_COLS, 
        experiment_name=f'exp051_sentiment_lexicon{seed}',
        verbose=False,  
    )
    exp051_sentiment_lexicon.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp051_sentiment_lexicon, name='exp051_sentiment_lexicon')
delta(exp051_sentiment_lexicon, best_result)

Esta variable no parece ser muy explicativa.

## 5.2 Análisis de emociones a través de transformers - Pysentimiento
Se utilizará la librería pysentimiento que permite hacer análisis de emociones.

In [ ]:
# --- Configuración ---
BATCH_SIZE = 32
CHECKPOINT_DIR = 'G:/Mi unidad/Maestría en Ciencia de Datos - Universidad Austral/Materias/11. Laboratorio de implementación II (jueves)/UA_MDM_Labo2_Grupo2/checkpoints_petfinder'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

pet_ids = df_fe['PetID'].tolist()
texts   = df_fe['DescriptionClean'].tolist()

In [ ]:
# --- Función con checkpoints por PetID ---
def procesar_con_checkpoints(analyzer, pet_ids, texts, task_name, batch_size=32):
    checkpoint_pkl = os.path.join(CHECKPOINT_DIR, f"{task_name}_results.pkl")

    # Cargar checkpoint si existe
    if os.path.exists(checkpoint_pkl):
        try:
            with open(checkpoint_pkl, 'rb') as f:
                results_dict = pickle.load(f)
            print(f"⚡ Checkpoint encontrado: {len(results_dict)} PetIDs ya procesados. Retomando...")
        except (EOFError, pickle.UnpicklingError):
            print("⚠️ Checkpoint corrupto, empezando desde cero...")
            results_dict = {}
    else:
        results_dict = {}
        print("🚀 Empezando desde cero...")

    # Filtrar solo los PetIDs que faltan procesar
    pendientes = [(pid, txt) for pid, txt in zip(pet_ids, texts) if pid not in results_dict]
    print(f"📋 Pendientes: {len(pendientes)} | Ya procesados: {len(results_dict)}")

    for i in tqdm(range(0, len(pendientes), batch_size), desc=task_name):
        batch_items  = pendientes[i:i + batch_size]
        batch_ids    = [item[0] for item in batch_items]
        batch_texts  = [item[1] if item[1] else "." for item in batch_items]

        try:
            batch_results = analyzer.predict(batch_texts)
            for pid, result in zip(batch_ids, batch_results):
                results_dict[pid] = result
        except Exception as e:
            print(f"\n⚠️ Error en batch {i}: {e}. Procesando uno por uno...")
            for pid, text in zip(batch_ids, batch_texts):
                try:
                    results_dict[pid] = analyzer.predict(text)
                except:
                    results_dict[pid] = None

        # Guardado atómico cada 10 batches
        if (i // batch_size) % 10 == 0:
            tmp_file = checkpoint_pkl + '.tmp'
            with open(tmp_file, 'wb') as f:
                pickle.dump(results_dict, f)
            os.replace(tmp_file, checkpoint_pkl)

    # Guardado final atómico
    tmp_file = checkpoint_pkl + '.tmp'
    with open(tmp_file, 'wb') as f:
        pickle.dump(results_dict, f)
    os.replace(tmp_file, checkpoint_pkl)

    print(f"✓ Total procesados: {len(results_dict)}")
    return results_dict

In [ ]:
# --- Emociones ---
print("Cargando modelo de emociones...")
emotion_analyzer = create_analyzer(task="emotion", lang="en")

print("Procesando emociones...")
emotion_results_dict = procesar_con_checkpoints(
    analyzer=emotion_analyzer,
    pet_ids=pet_ids,
    texts=texts,
    task_name="emotion"
)

# Detectar emociones disponibles dinámicamente
_sample = emotion_analyzer.predict("test")
emotions = list(_sample.probas.keys())
print(f"Emociones detectadas: {emotions}")

# Asignar al dataframe por PetID (seguro ante reordenamientos)
df_fe['emotion_label'] = df_fe['PetID'].map(
    lambda pid: emotion_results_dict[pid].output if emotion_results_dict.get(pid) else None
)
for emo in emotions:
    df_fe[f'emotion_{emo}'] = df_fe['PetID'].map(
        lambda pid: emotion_results_dict[pid].probas.get(emo, 0) if emotion_results_dict.get(pid) else None
    )

print("✓ Emociones asignadas al dataframe")

In [ ]:
# Definir el mapa de emociones
emotion_map = {
    'others': 0,
    'joy': 1,
    'sadness': 2,
    'anger': 3,
    'fear': 4,
    'surprise': 5,
    'disgust': 6
}

# Aplicar a la columna
df_fe['emotion_label_encoded'] = df_fe['emotion_label'].map(emotion_map)

NEW_INT = ['emotion_label_encoded']
CURRENT_FEATURES_52 = CURRENT_FEATURES_42_reemplazo + AUX_COLS + NEW_INT
X_52 = df_fe[CURRENT_FEATURES_52]

print(f'Features nuevas (5.2): {NEW_INT}')
print(f'Total features: {len(CURRENT_FEATURES_52)}')
print('\n' + '='*55)
print('EXPERIMENTO 5.2: + Emotion Analysis (Transformers)')
print('='*55)

exp052_emotion = []

for seed in SEEDS:
    r = run_cv(
        X=X_52,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals + NEW_INT,
        agg_config=AGG_RESCUER,
        agg_smoothing=AGG_SMOOTHING,       # rescatistas con <n publicaciones tiran a la media
        aux_cols=AUX_COLS, 
        experiment_name=f'exp052_emotion{seed}',
        verbose=False,  
    )
    exp052_emotion.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp052_emotion, name='exp052_emotion')
delta(exp052_emotion, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp052_emotion], title='exp052_emotion')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp052_emotion])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=25, title='Feature Importance — exp052_emotion')


La gran mayoría de los registros tienen como sentiment asignado 'others'. Tal vez sea necesario forzar a que se asignen otras etiquetas para mejorar la señal.

In [ ]:
def get_best_emotion(pid, results_dict):
    res = results_dict.get(pid)
    if not res or not hasattr(res, 'probas'):
        return None
    
    # Filtramos 'others' de las probabilidades
    filtered_probas = {k: v for k, v in res.probas.items() if k != 'others'}
    
    if not filtered_probas:
        return None
        
    # Retornamos la llave con el valor máximo
    return max(filtered_probas, key=filtered_probas.get)

# 1. Asignar la nueva etiqueta predominante (sin 'others')
df_fe['emotion_label'] = df_fe['PetID'].map(
    lambda pid: get_best_emotion(pid, emotion_results_dict)
)

# 2. Las columnas de probabilidades individuales se mantienen igual 
# (o puedes decidir si quieres borrarlas después)
for emo in emotions:
    df_fe[f'emotion_{emo}'] = df_fe['PetID'].map(
        lambda pid: emotion_results_dict[pid].probas.get(emo, 0) if emotion_results_dict.get(pid) else None
    )

df_fe['emotion_label'].value_counts()

In [ ]:
# Definir el mapa de emociones
emotion_map = {
    'others': 0,
    'joy': 1,
    'sadness': 2,
    'anger': 3,
    'fear': 4,
    'surprise': 5,
    'disgust': 6
}

# Aplicar a la columna
df_fe['emotion_label_encoded'] = df_fe['emotion_label'].map(emotion_map)

NEW_INT = ['emotion_label_encoded']
CURRENT_FEATURES_52_so = CURRENT_FEATURES_42_reemplazo + AUX_COLS + NEW_INT
X_52_so = df_fe[CURRENT_FEATURES_52_so]

print(f'Features nuevas (5.2): {NEW_INT}')
print(f'Total features: {len(CURRENT_FEATURES_52_so)}')
print('\n' + '='*55)
print('EXPERIMENTO 5.2: + Emotion Analysis sin Others (Transformers)')
print('='*55)

exp052_emotion_so = []

for seed in SEEDS:
    r = run_cv(
        X=X_52_so,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals + NEW_INT,
        agg_config=AGG_RESCUER,
        agg_smoothing=AGG_SMOOTHING,       # rescatistas con <n publicaciones tiran a la media
        aux_cols=AUX_COLS, 
        experiment_name=f'exp052_emotion_so{seed}',
        verbose=False,  
    )
    exp052_emotion_so.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp052_emotion_so, name='exp052_emotion_so')
delta(exp052_emotion_so, best_result)

Esto tampoco funcionó.. 

## 5.3 Análisis de sentimiento a través de transformers - Pysentimiento
Se utilizará la librería pysentimiento que, además de análisis de emociones, permite hacer análisis de sentimiento.

In [ ]:
# --- Sentimiento ---
print("Cargando modelo de sentimiento...")
sent_analyzer = create_analyzer(task="sentiment", lang="en")

print("Procesando sentimiento...")
sent_results_dict = procesar_con_checkpoints(
    analyzer=sent_analyzer,
    pet_ids=pet_ids,
    texts=texts,
    task_name="sentiment"
)

# Asignar al dataframe por PetID
df_fe['pysent_label'] = df_fe['PetID'].map(
    lambda pid: sent_results_dict[pid].output if sent_results_dict.get(pid) else None
)
df_fe['pysent_pos'] = df_fe['PetID'].map(
    lambda pid: sent_results_dict[pid].probas.get('POS', 0) if sent_results_dict.get(pid) else None
)
df_fe['pysent_neg'] = df_fe['PetID'].map(
    lambda pid: sent_results_dict[pid].probas.get('NEG', 0) if sent_results_dict.get(pid) else None
)
df_fe['pysent_neu'] = df_fe['PetID'].map(
    lambda pid: sent_results_dict[pid].probas.get('NEU', 0) if sent_results_dict.get(pid) else None
)

print("✓ Sentimiento asignado al dataframe")

In [ ]:
# Definir el mapa de emociones
pysent_map = {
    'POST': 1,
    'NEU': 0,
    'NEG': -1,
}

# Aplicar a la columna
df_fe['pysent_label_encoded'] = df_fe['pysent_label'].map(pysent_map)

NEW_INT = ['pysent_label_encoded']
CURRENT_FEATURES_53 = CURRENT_FEATURES_42_reemplazo + AUX_COLS + NEW_INT
X_53 = df_fe[CURRENT_FEATURES_53]

print(f'Features nuevas (5.3): {NEW_INT}')
print(f'Total features: {len(CURRENT_FEATURES_53)}')
print('\n' + '='*55)
print('EXPERIMENTO 5.3: + Sentiment Analysis (Transformers)')
print('='*55)

exp053_sentiment = []

for seed in SEEDS:
    r = run_cv(
        X=X_53,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals + NEW_INT,
        agg_config=AGG_RESCUER,
        agg_smoothing=AGG_SMOOTHING,       # rescatistas con <n publicaciones tiran a la media
        aux_cols=AUX_COLS, 
        experiment_name=f'exp053_sentiment{seed}',
        verbose=False,  
    )
    exp053_sentiment.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp053_sentiment, name='exp053_sentiment')
delta(exp053_sentiment, best_result)

## 5.4 Extracción de Embeddings con Transformes - SentenceTransformer
Se realizará un proceso de extracción de Embeddings, aplicación de PCA para reducción de dimensionalidad, e incorporación de los N componentes más explicativos.

In [ ]:
CHECKPOINT_DIR = 'G:/Mi unidad/Maestría en Ciencia de Datos - Universidad Austral/Materias/11. Laboratorio de implementación II (jueves)/UA_MDM_Labo2_Grupo2/checkpoints_petfinder'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 64

# --- Cargar modelo ---
print("Cargando MiniLM...")
text_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✓ Modelo cargado")

def extraer_embeddings_texto(model, pet_ids, texts, checkpoint_dir, batch_size=64):
    checkpoint_pkl = os.path.join(checkpoint_dir, "text_embeddings.pkl")

    # Cargar checkpoint si existe
    if os.path.exists(checkpoint_pkl):
        try:
            with open(checkpoint_pkl, 'rb') as f:
                embeddings_dict = pickle.load(f)
            print(f"⚡ Checkpoint encontrado: {len(embeddings_dict)} PetIDs ya procesados. Retomando...")
        except (EOFError, pickle.UnpicklingError):
            print("⚠️ Checkpoint corrupto, empezando desde cero...")
            embeddings_dict = {}
    else:
        embeddings_dict = {}
        print("🚀 Empezando desde cero...")

    # Filtrar pendientes
    pendientes = [(pid, txt) for pid, txt in zip(pet_ids, texts) if pid not in embeddings_dict]
    print(f"📋 Pendientes: {len(pendientes)} | Ya procesados: {len(embeddings_dict)}")

    for i in tqdm(range(0, len(pendientes), batch_size), desc="text_embeddings"):
        batch_items  = pendientes[i:i + batch_size]
        batch_ids    = [item[0] for item in batch_items]
        batch_texts  = [item[1] if item[1] else "." for item in batch_items]

        try:
            batch_embs = model.encode(batch_texts, convert_to_numpy=True, show_progress_bar=False)
            for pid, emb in zip(batch_ids, batch_embs):
                embeddings_dict[pid] = emb
        except Exception as e:
            print(f"\n⚠️ Error en batch {i}: {e}. Procesando uno por uno...")
            for pid, text in zip(batch_ids, batch_texts):
                try:
                    embeddings_dict[pid] = model.encode(text, convert_to_numpy=True)
                except:
                    embeddings_dict[pid] = None

        # Guardado atómico cada 10 batches
        if (i // batch_size) % 10 == 0:
            tmp_file = checkpoint_pkl + '.tmp'
            with open(tmp_file, 'wb') as f:
                pickle.dump(embeddings_dict, f)
            os.replace(tmp_file, checkpoint_pkl)

    # Guardado final atómico
    tmp_file = checkpoint_pkl + '.tmp'
    with open(tmp_file, 'wb') as f:
        pickle.dump(embeddings_dict, f)
    os.replace(tmp_file, checkpoint_pkl)

    print(f"✓ Total procesados: {len(embeddings_dict)}")
    return embeddings_dict

# --- Extraer ---
pet_ids = df_fe['PetID'].tolist()
texts   = df_fe['DescriptionClean'].tolist()

text_embeddings_dict = extraer_embeddings_texto(text_model, pet_ids, texts, CHECKPOINT_DIR)

# --- Convertir a matriz alineada con df_fe ---
text_embeddings_matrix = np.array([
    text_embeddings_dict[pid] if text_embeddings_dict.get(pid) is not None
    else np.zeros(384)
    for pid in df_fe['PetID']
])
print(f"✓ Shape matriz de texto: {text_embeddings_matrix.shape}")  # (15000, 384)

Contando con aproximadamente 30 features tabulares, extraer más de 40 features de Embeddings puede generar interferencia en el modelo. Se evaluará el QWK obtenido al incorporar las primeras 20, 30 y 40 features.

In [ ]:
pca_full = PCA(random_state = SEED).fit(text_embeddings_matrix)
plt.plot(np.cumsum(pca_full.explained_variance_ratio_))
plt.xlabel("Componentes")
plt.ylabel("Varianza acumulada")
plt.axhline(0.70, color='r', linestyle='--', label='70%')
plt.legend()
plt.show()

In [ ]:
pca = PCA(n_components=40, random_state=SEED)
text_reduced = pca.fit_transform(text_embeddings_matrix)

for n in [20, 30, 40]:
    varianza = pca.explained_variance_ratio_[:n].sum()
    print(f"Varianza explicada con {n} features: {varianza:.2%}")

def make_text_df(matrix, n, index):
    return pd.DataFrame(
        matrix[:, :n],
        index=index,
        columns=[f"text_embedding_{i}" for i in range(n)]
    )

text_reduced_20 = make_text_df(text_reduced, 20, df.index)
text_reduced_30 = make_text_df(text_reduced, 30, df.index)
text_reduced_40 = make_text_df(text_reduced, 40, df.index)

df_fe = pd.concat([df_fe, text_reduced_40], axis=1)

NEW_INT = list(text_reduced_30.columns)
CURRENT_FEATURES_54 = CURRENT_FEATURES_42_reemplazo + AUX_COLS + NEW_INT
X_54 = df_fe[CURRENT_FEATURES_54]

print(f'Features nuevas (5.4): {NEW_INT}')
print(f'Total features: {len(CURRENT_FEATURES_54)}')
print('\n' + '='*55)
print('EXPERIMENTO 5.4: + Emotion Analysis (Transformers)')
print('='*55)

exp054_text_embed = []

for seed in SEEDS:
    r = run_cv(
        X=X_54,
        y=y,
        params=BASELINE_PARAMS,
        seed = seed,
        cat_features=categoricals,
        agg_config=AGG_RESCUER,
        agg_smoothing=AGG_SMOOTHING,       # rescatistas con <n publicaciones tiran a la media
        aux_cols=AUX_COLS, 
        experiment_name=f'exp054_text_embed{seed}',
        verbose=False,  
    )
    exp054_text_embed.append(r)
    print(f"Seed {seed}: QWK = {r['qwk_mean']:.4f} ± {r['qwk_std']:.4f}")

tracker.log(exp054_text_embed, name='exp054_text_embed')
delta(exp054_text_embed, best_result)

In [ ]:
plot_results(y, [r['oof_preds'] for r in exp054_text_embed], title='exp054_text_embed')

all_feat_imp = pd.concat([r['feat_imp'] for r in exp054_text_embed])

feat_imp_combined = (
    all_feat_imp
    .groupby('feature')
    .agg(
        gain      =('gain',      'mean'),
        gain_std  =('gain',      'std'),   # std entre seeds (no entre folds)
        split     =('split',     'mean'),
        split_std =('split',     'std'),
    )
    .reset_index()
    .sort_values('gain', ascending=False)
)

plot_importance(feat_imp_combined, top_n=40, title='Feature Importance — exp054_text_embed')


Los mejores resultados se obtienen al utilizar 30 componentes. 
Se incorporarán estas variables al modelo.

---
# 6 Análisis de Imágenes

Se utilizará el modelo MobileNetV2 para extraer embeddings de las imágenes, se realizará un promedio de embeddings por PetID, luego se aplicará PCA para extraer las N features que permitan capturar variabilidad de las imágenes, por último se entrenará un nuevo experimento del modelo tabular para evaluar si mejora el QWK.
Por falta de GPU, el proceso se corrió por separado en Google Colab. Un proceso que en local hubiera tardado 22 horas en correr, en Colab con GPUx4 demoró 2 horas y media.

In [ ]:
# ── Celda 2: Configuración ──────────────────────────────────
import os
import numpy as np
import pandas as pd
import pickle
import time
from tqdm.auto import tqdm
from PIL import Image
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

#DRIVE_BASE_DIR = '/content/drive/MyDrive/Maestría en Ciencia de Datos - Universidad Austral/Materias/11. Laboratorio de implementación II (jueves)/UA_MDM_Labo2_Grupo2'
DRIVE_BASE_DIR = 'G:/Mi unidad/Maestría en Ciencia de Datos - Universidad Austral/Materias/11. Laboratorio de implementación II (jueves)/UA_MDM_Labo2_Grupo2'
TABULAR_DIR    = os.path.join(DRIVE_BASE_DIR, 'input/petfinder-adoption-prediction/train/train.csv')
IMAGE_DIR      = os.path.join(DRIVE_BASE_DIR, 'input/petfinder-adoption-prediction/train_images')
CHECKPOINT_DIR = os.path.join(DRIVE_BASE_DIR, 'checkpoints_petfinder')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
IMG_SIZE    = (160, 160)
BATCH_SIZE  = 64   
NUM_WORKERS = 2    

CHECKPOINT_EVERY = 50
df_fe = pd.read_csv(TABULAR_DIR)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Celda 3: Modelo ─────────────────────────────────────────
print("Cargando MobileNetV2...")
mobilenet = models.mobilenet_v2(weights='IMAGENET1K_V1')
mobilenet.classifier = torch.nn.Identity()
mobilenet.eval().to(DEVICE)
print("✓ Modelo cargado y enviado a", DEVICE)
 
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
# ── Celda 4: Dataset ────────────────────────────────────────
class PetImageDataset(Dataset):
    def __init__(self, image_dir, pet_ids, transform):
        self.transform = transform
        self.samples   = []  # (pet_id, path)
 
        for pid in pet_ids:
            img_idx = 1
            while True:
                path = os.path.join(image_dir, f"{pid}-{img_idx}.jpg")
                if not os.path.exists(path):
                    break
                self.samples.append((pid, path))
                img_idx += 1
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        pet_id, path = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            return pet_id, self.transform(img), True
        except Exception:
            return pet_id, torch.zeros(3, *IMG_SIZE), False

In [ ]:
# ── Celda 5: Helpers de checkpoint ──────────────────────────
CHECKPOINT_PKL = os.path.join(CHECKPOINT_DIR, 'img_embeddings.pkl')
 
def cargar_checkpoint():
    if not os.path.exists(CHECKPOINT_PKL):
        print("No se encontró checkpoint previo. Empezando desde cero.")
        return {}
    try:
        with open(CHECKPOINT_PKL, 'rb') as f:
            data = pickle.load(f)
        print(f"⚡ Checkpoint cargado: {len(data)} PetIDs ya procesados.")
        return data
    except (EOFError, pickle.UnpicklingError):
        print("⚠️  Checkpoint corrupto. Empezando desde cero.")
        return {}
 
def guardar_checkpoint(embeddings_por_pet):
    """Escritura atómica: tmp → rename, para no corromper si se corta."""
    tmp = CHECKPOINT_PKL + '.tmp'
    with open(tmp, 'wb') as f:
        pickle.dump(embeddings_por_pet, f, protocol=4)
    os.replace(tmp, CHECKPOINT_PKL)
    print(f"  💾 Checkpoint guardado ({len(embeddings_por_pet)} PetIDs)")
 
 

In [ ]:
# ── Celda 6: Extracción principal ───────────────────────────
def extraer_embeddings(pet_ids):
    embeddings_por_pet = cargar_checkpoint()
 
    pendientes = [pid for pid in pet_ids if pid not in embeddings_por_pet]
    print(f"📋 Pendientes: {len(pendientes)} / {len(pet_ids)} PetIDs")
 
    if not pendientes:
        print("✓ Nada que procesar.")
        return embeddings_por_pet
 
    dataset = PetImageDataset(IMAGE_DIR, pendientes, transform)
    loader  = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda'),
    )
    print(f"   Imágenes a procesar: {len(dataset)}")
 
    # Acumula embeddings por pet durante el recorrido
    acum = {}   # pet_id -> [embedding, ...]
 
    t0 = time.time()
    with torch.no_grad():
        for batch_idx, (pet_ids_batch, imgs, validos) in enumerate(
            tqdm(loader, desc="Embeddings", unit="batch")
        ):
            imgs_validas = imgs[validos].to(DEVICE)
            pids_validos = [pid for pid, v in zip(pet_ids_batch, validos) if v]
 
            if len(imgs_validas) == 0:
                continue
 
            embs = mobilenet(imgs_validas).cpu().numpy()  # (B, 1280)
 
            for pid, emb in zip(pids_validos, embs):
                acum.setdefault(pid, []).append(emb)
 
            # ── Checkpoint parcial ───────────────────────────────────
            # Cada CHECKPOINT_EVERY batches, consolida lo acumulado y guarda.
            # Si Colab se corta acá, solo se pierde el batch en curso.
            if (batch_idx + 1) % CHECKPOINT_EVERY == 0:
                _consolidar_y_guardar(acum, embeddings_por_pet)
                acum.clear()
                elapsed = time.time() - t0
                print(f"  ⏱  {elapsed/60:.1f} min transcurridos | "
                      f"{len(embeddings_por_pet)} PetIDs guardados")
 
    # Consolidar lo que quedó al final
    _consolidar_y_guardar(acum, embeddings_por_pet)
 
    # Pets sin ninguna imagen válida
    for pid in pendientes:
        if pid not in embeddings_por_pet:
            embeddings_por_pet[pid] = {'embedding': None, 'n_imagenes': 0}
 
    guardar_checkpoint(embeddings_por_pet)
    print(f"\n✓ Procesamiento completo. Total: {len(embeddings_por_pet)} PetIDs")
    return embeddings_por_pet
 
 
def _consolidar_y_guardar(acum, embeddings_por_pet):
    """Promedia los embeddings acumulados y los vuelca al dict principal."""
    for pid, emb_list in acum.items():
        arr = np.stack(emb_list)
        embeddings_por_pet[pid] = {
            'embedding':   arr.mean(axis=0),
            'n_imagenes':  len(arr),
        }
    if acum:
        guardar_checkpoint(embeddings_por_pet)
 

In [ ]:
# ── Celda 6: Extracción principal ───────────────────────────
def extraer_embeddings(pet_ids):
    embeddings_por_pet = cargar_checkpoint()
 
    pendientes = [pid for pid in pet_ids if pid not in embeddings_por_pet]
    print(f"📋 Pendientes: {len(pendientes)} / {len(pet_ids)} PetIDs")
 
    if not pendientes:
        print("✓ Nada que procesar.")
        return embeddings_por_pet
 
    dataset = PetImageDataset(IMAGE_DIR, pendientes, transform)
    loader  = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == 'cuda'),
    )
    print(f"   Imágenes a procesar: {len(dataset)}")
 
    # Acumula embeddings por pet durante el recorrido
    acum = {}   # pet_id -> [embedding, ...]
 
    t0 = time.time()
    with torch.no_grad():
        for batch_idx, (pet_ids_batch, imgs, validos) in enumerate(
            tqdm(loader, desc="Embeddings", unit="batch")
        ):
            imgs_validas = imgs[validos].to(DEVICE)
            pids_validos = [pid for pid, v in zip(pet_ids_batch, validos) if v]
 
            if len(imgs_validas) == 0:
                continue
 
            embs = mobilenet(imgs_validas).cpu().numpy()  # (B, 1280)
 
            for pid, emb in zip(pids_validos, embs):
                acum.setdefault(pid, []).append(emb)
 
            # ── Checkpoint parcial ───────────────────────────────────
            # Cada CHECKPOINT_EVERY batches, consolida lo acumulado y guarda.
            # Si Colab se corta acá, solo se pierde el batch en curso.
            if (batch_idx + 1) % CHECKPOINT_EVERY == 0:
                _consolidar_y_guardar(acum, embeddings_por_pet)
                acum.clear()
                elapsed = time.time() - t0
                print(f"  ⏱  {elapsed/60:.1f} min transcurridos | "
                      f"{len(embeddings_por_pet)} PetIDs guardados")
 
    # Consolidar lo que quedó al final
    _consolidar_y_guardar(acum, embeddings_por_pet)
 
    # Pets sin ninguna imagen válida
    for pid in pendientes:
        if pid not in embeddings_por_pet:
            embeddings_por_pet[pid] = {'embedding': None, 'n_imagenes': 0}
 
    guardar_checkpoint(embeddings_por_pet)
    print(f"\n✓ Procesamiento completo. Total: {len(embeddings_por_pet)} PetIDs")
    return embeddings_por_pet
 
 
def _consolidar_y_guardar(acum, embeddings_por_pet):
    """Promedia los embeddings acumulados y los vuelca al dict principal."""
    for pid, emb_list in acum.items():
        arr = np.stack(emb_list)
        embeddings_por_pet[pid] = {
            'embedding':   arr.mean(axis=0),
            'n_imagenes':  len(arr),
        }
    if acum:
        guardar_checkpoint(embeddings_por_pet)
 

In [ ]:
# ── Celda 7: Ejecutar ─────────────────────────────────────── 
pet_ids       = df_fe['PetID'].tolist()
img_embeddings = extraer_embeddings(pet_ids)
 
df_fe['img_n_imagenes'] = df_fe['PetID'].map(
    lambda pid: img_embeddings[pid]['n_imagenes'] if img_embeddings.get(pid) else 0
)


---
# 5. Tuning de hiperparámetros con Optuna

Búsqueda bayesiana sobre el espacio de hiperparámetros de LightGBM.
Se usa 3-fold durante la búsqueda (velocidad) y se valida el mejor trial con 5-fold.

In [ ]:
# X_final usa el mejor conjunto de features encontrado en FE
FINAL_FEATURES = CURRENT_FEATURES_42_reemplazo
FINAL_CAT_FEATURES = categoricals
FINAL_AGG_CONFIG = AGG_RESCUER
FINAL_AGG_SMOOTHING = AGG_SMOOTHING
FINAL_AUX_COLS = AUX_COLS
#FINAL_TE_COLS = TE_COLS
#FINAL_TE_SMOOTHING = 'auto'
X_final = df_fe[FINAL_FEATURES + FINAL_AUX_COLS].copy()

def objective(trial):
    params = {
    # ── Fijos ────────────────────────────────────────────────────────────────
    'objective'   : 'multiclass',
    'num_class'   : 5,
    'metric'      : 'multi_logloss',
    'verbose'     : -1,
    'n_estimators': 4000, 
    'learning_rate'    : trial.suggest_float('learning_rate',     0.001, 0.05, log=True),
    'num_leaves'       : trial.suggest_int('num_leaves',          31, 150),
    'max_depth'        : trial.suggest_int('max_depth',           4, 9),
    'min_child_samples': trial.suggest_int('min_child_samples',   20, 150),
    'subsample'        : trial.suggest_float('subsample',         0.3, 1.0),
    'bagging_freq'     : trial.suggest_int('bagging_freq',        1, 7),
    'colsample_bytree' : trial.suggest_float('colsample_bytree',  0.5, 1.0),
    #'reg_alpha'        : trial.suggest_float('reg_alpha',         1e-8, 1.0, log=True),
    #'reg_lambda'       : trial.suggest_float('reg_lambda',        1e-8, 1.0, log=True),
    'min_sum_hessian_in_leaf': trial.suggest_float('min_sum_hessian_in_leaf', 1e-3, 10.0, log=True),
}

    results = run_cv(
        X = X_final,   # mismas features que en FE
        y = y,
        params = params,
        cat_features = FINAL_CAT_FEATURES,   # los mismos que usaste en FE
        n_folds = 3,                   # 3 folds para velocidad
        verbose = False,
        experiment_name = f'trial_{trial.number}',
        #te_cols=FINAL_TE_COLS,       # ← calculado dentro del fold, sin leakage
        #te_smoothing=FINAL_TE_SMOOTHING,
        agg_config=FINAL_AGG_CONFIG,
        agg_smoothing=FINAL_AGG_SMOOTHING,
        aux_cols=FINAL_AUX_COLS,
    )

    # Pruning por fold (necesitás exponer fold_qwk de run_cv)
    for fold, qwk_fold in enumerate(results['fold_qwk']):
        trial.report(qwk_fold, fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return results['qwk_mean']


print('Función objetivo definida.')

In [ ]:

os.makedirs('G:/Mi unidad/Maestría en Ciencia de Datos - Universidad Austral/Materias/11. Laboratorio de implementación II (jueves)/UA_MDM_Labo2_Grupo2/cache', exist_ok=True)
BO_VERSION = 'v1_target_encoding'  # ← cambiás esto para cada versión
BO_CACHE_PATH = f'G:/Mi unidad/Maestría en Ciencia de Datos - Universidad Austral/Materias/11. Laboratorio de implementación II (jueves)/UA_MDM_Labo2_Grupo2/cache/study_{BO_VERSION}.pkl'

if os.path.exists(BO_CACHE_PATH):
    study = joblib.load(BO_CACHE_PATH)
    print(f'Study cargado desde caché: {BO_CACHE_PATH}')
    print(f'Trials completados: {len(study.trials)}')

else:
    N_TRIALS = 80
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
    )
    print(f'Iniciando búsqueda: {N_TRIALS} trials...')
    t0 = time.time()
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    print(f'\nTiempo total: {(time.time()-t0)/60:.1f} min')

    joblib.dump(study, BO_CACHE_PATH)
    print(f'Study guardado en: {BO_CACHE_PATH}')

print(f'\nMejor QWK (3-fold): {study.best_value:.4f}')
print('Mejores hiperparámetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

trials_df = study.trials_dataframe()
completed = trials_df[trials_df['state'] == 'COMPLETE'].copy()
completed['best_so_far'] = completed['value'].cummax()

axes[0].plot(completed.index, completed['value'],
             alpha=0.4, color=PALETTE[2], linewidth=0.8, label='Trial')
axes[0].plot(completed.index, completed['best_so_far'],
             color=PALETTE[0], linewidth=2, label='Mejor hasta ahora')
axes[0].set_title('Evolución del QWK durante la búsqueda')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('QWK (3-fold)')
axes[0].legend()

try:
    imp = optuna.importance.get_param_importances(study)
    imp_df = pd.DataFrame(list(imp.items()), columns=['Param','Importance']).sort_values('Importance')
    axes[1].barh(imp_df['Param'], imp_df['Importance'], color=PALETTE[1], edgecolor='white')
    axes[1].set_title('Importancia de hiperparámetros')
    axes[1].set_xlabel('Importancia relativa')
except Exception as e:
    axes[1].text(0.5, 0.5, f'No disponible:\n{e}', ha='center', va='center')
    axes[1].axis('off')

plt.suptitle('Optuna — Proceso de búsqueda', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Validación final del mejor trial con 5-fold
BEST_PARAMS = {
    'objective'   : 'multiclass',
    'num_class'   : 5,
    'metric'      : 'multi_logloss',
    'verbose'     : -1,
    'n_estimators': 3000,   # más estimadores con el LR bajo del tuning
    **study.best_params
}

print('='*55)
print('EXPERIMENTO FINAL: Mejor FE + Hiperparámetros tuneados (5-fold)')
print('='*55)

result_tuned = run_cv(
    X = X_final,   # mismas features que en FE
    y = y,
    params = BEST_PARAMS,
    cat_features = FINAL_CAT_FEATURES,   # los mismos que usaste en FE
    n_folds = 5,                   # 3 folds para velocidad
    verbose = False,
    experiment_name='09_tuned',
    #te_cols=FINAL_TE_COLS,       # ← calculado dentro del fold, sin leakage
    #te_smoothing=FINAL_TE_SMOOTHING,
    agg_config=FINAL_AGG_CONFIG,
    agg_smoothing=FINAL_AGG_SMOOTHING,
    aux_cols=FINAL_AUX_COLS,
    )


tracker.log(result_tuned)
delta(result_tuned, best_result)

In [ ]:
plot_results(y, result_tuned['oof_preds'], title='Modelo Tuneado — Resultado Final')
plot_importance(result_tuned['feat_imp'], top_n=50, title='Feature Importance — Modelo Tuneado')

---
# 6. Resumen final y análisis de errores

In [ ]:
print('\n' + '='*55)
print('TABLA RESUMEN — TODOS LOS EXPERIMENTOS')
print('='*55)
display(tracker.table())
tracker.plot()

In [ ]:
preds_final = result_tuned['oof_preds'].astype(int)
errors = df.copy()
errors['pred']       = preds_final
errors['error']      = (errors['pred'] != errors[TARGET]).astype(int)
errors['error_size'] = (errors['pred'] - errors[TARGET]).abs()

print('Distribución de errores por magnitud:')
print(errors['error_size'].value_counts().sort_index().to_string())
print(f'\nError de 0 clases (correcto): {(errors["error_size"]==0).mean()*100:.1f}%')
print(f'Error de 1 clase:             {(errors["error_size"]==1).mean()*100:.1f}%')
print(f'Error de 2+ clases:           {(errors["error_size"]>=2).mean()*100:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

err_by_class = errors.groupby(TARGET)['error'].mean() * 100
axes[0].bar(err_by_class.index, err_by_class.values, color=PALETTE[3], edgecolor='white')
axes[0].set_title('Tasa de error por clase real')
axes[0].set_xlabel('AdoptionSpeed real')
axes[0].set_ylabel('% error')
for i, v in enumerate(err_by_class.values):
    axes[0].text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=9)

err_by_type = errors.groupby('Type')['error'].mean() * 100
err_by_type.index = ['Perro', 'Gato']
axes[1].bar(err_by_type.index, err_by_type.values, color=PALETTE[1], edgecolor='white')
axes[1].set_title('Tasa de error por tipo')
axes[1].set_ylabel('% error')
for i, v in enumerate(err_by_type.values):
    axes[1].text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=9)

axes[2].boxplot(
    [errors[errors['error_size']==k]['Age'].dropna() for k in [0,1,2,3,4]],
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2),
    boxprops=dict(facecolor=PALETTE[0], alpha=0.6)
)
axes[2].set_xticklabels(['±0','±1','±2','±3','±4'])
axes[2].set_title('Edad según magnitud del error')
axes[2].set_xlabel('Clases de error')
axes[2].set_ylabel('Age (meses)')

plt.suptitle('Análisis de errores — Modelo Tuneado', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 7. Hiperparámetros finales y conclusiones

In [ ]:
print('Hiperparámetros del modelo final:')
for k, v in BEST_PARAMS.items():
    if k not in ['objective','num_class','metric','verbose']:
        print(f'  {k}: {v}')

print(f'\nFeatures en el modelo final ({len(FINAL_FEATURES)}):')
for f in sorted(FINAL_FEATURES):
    print(f'  {f}')

### Conclusiones

**Baseline** establece el piso de QWK con las features originales y HealthScore.

**Feature Engineering** — los bloques que típicamente aportan más en este dataset:
- Target encoding de `Breed1`, `State` y `RescuerID` (captura señales de alta cardinalidad)
- Transformaciones logarítmicas de `Fee`, `Age` y `PhotoAmt` (reduce el impacto de outliers)
- SVD sobre `Description` (captura semántica del texto)
- Aggregations por rescatista (captura el "estilo" de publicación)

**Tuning con Optuna** — el espacio más sensible es `num_leaves` y `learning_rate`.
`min_child_samples` controla el overfitting más que la regularización L1/L2.

**Threshold optimization** — en problemas ordinales como este, optimizar los umbrales
post-hoc casi siempre da un pequeño delta positivo de QWK sin costo adicional.

**Próximos pasos:** modelo de imágenes con embeddings preentrenados → fusión con el modelo tabular.

In [ ]:
# ── 1. Cargar y preprocesar test (igual que train) ────────────────────────────
test = pd.read_csv(os.path.join(input_path, 'test/test.csv'))
df_test = test.copy()

# Limpieza de Name
df_test['Name'] = df_test['Name'].astype(str).str.lower().str.strip()
mask_name = (
    df_test['Name'].str.contains('|'.join(invalid_kw), na=False) |
    (df_test['Name'].str.len() <= 1) |
    df_test['Name'].str.match(r'^[^a-zA-Z0-9]+$') |
    df_test['Name'].str.isnumeric() |
    (df_test['Name'] == 'nan')
)
df_test.loc[mask_name, 'Name'] = np.nan

# Limpieza de Description
mask_kw    = df_test['Description'].str.lower().str.contains('|'.join(generic_phrases), na=False)
mask_short = df_test['Description'].str.len() < 40
df_test.loc[mask_kw & mask_short, 'Description'] = np.nan
df_test['Description'] = df_test['Description'].str.strip()
df_test.loc[df_test['Description'].str.len() < 5, 'Description'] = np.nan

# Merge con breeds, colors, states (igual que en train)
df_test['Type']   = df_test['Type'].astype(int)
df_test['Breed1'] = df_test['Breed1'].astype(int)
df_test['Breed2'] = df_test['Breed2'].astype(int)
df_test = df_test.merge(breeds_, left_on=['Type','Breed1'], right_on=['Type','BreedID'], how='left')
df_test.rename(columns={'BreedName': 'Breed1Name'}, inplace=True)
df_test.drop(columns=['BreedID'], inplace=True)
df_test = df_test.merge(breeds_, left_on=['Type','Breed2'], right_on=['Type','BreedID'], how='left')
df_test.rename(columns={'BreedName': 'Breed2Name'}, inplace=True)
df_test.drop(columns=['BreedID'], inplace=True)
df_test['StateName']   = df_test['State'].map(state_map)
df_test['Color1Label'] = df_test['Color1'].map(color_map)

# ── 3. Aplicar el mismo feature engineering que en df_fe ─────────────────────
# (todas las transformaciones que aplicaste para construir df_fe,
#  pero sobre df_test — flags, logs, bins, HealthScore, etc.)
# Por ejemplo:

df_test['HasPhoto']     = (df_test['PhotoAmt'] > 0).astype(int)
df_test['HasVideo']     = (df_test['VideoAmt'] > 0).astype(int)
df_test['PhotoAmt_log'] = np.log1p(df_test['PhotoAmt'])
df_test['VideoAmt_log'] = np.log1p(df_test['VideoAmt'])
df_test['StatePop']       = df_test['State'].map(malaysia_stats.set_index('StateID')['population'].to_dict())
df_test['StateIncomeMedian'] = df_test['State'].map(malaysia_stats.set_index('StateID')['income_median'].to_dict())
df_test['StateMarriageRate'] = df_test['State'].map(malaysia_stats.set_index('StateID')['marriage_rate'].to_dict())
df_test['HasDesc']       = df_test['Description'].notna().astype(int)
df_test['DescLength']    = df_test['Description'].str.len().fillna(0)
df_test['DescWordCount'] = df_test['Description'].fillna('').str.split().str.len()
df_test['DescRatio']     = df_test['DescLength'] / df_test['DescWordCount'].replace(0, np.nan)
df_test['VaccYes']   = (df_test['Vaccinated']  == 1).astype(int)
df_test['DewormYes'] = (df_test['Dewormed']    == 1).astype(int)
df_test['SterYes']   = (df_test['Sterilized']  == 1).astype(int)
df_test['Healthy']   = (df_test['Health']  == 1).astype(int)

health_map = {0: np.nan, 1: 1.0, 2: 0.5, 3: 0.0}
df_test['HealthScore'] = ( (df_test['VaccYes'] + df_test['DewormYes'] + df_test['SterYes'])/3 * 0.4 ) + (df_test['Health'].replace(health_map) * 0.6)

"""
# ── 2. Calcular las aggregations fitteadas sobre TODO el train ────────────────
# Esto reemplaza lo que run_cv hace dentro de cada fold
for group_col, aggs in FINAL_AGG_CONFIG.items():
    agg_kwargs = {
        new_name: pd.NamedAgg(column=src_col, aggfunc=func)
        for new_name, (src_col, func) in aggs.items()
    }
    agg_df = df_fe.groupby(group_col).agg(**agg_kwargs).reset_index()
    
    # Mergear a test — rescatistas nuevos en test → fillna con media de train
    df_test = df_test.merge(agg_df, on=group_col, how='left')
    for new_name in aggs.keys():
        df_test[new_name].fillna(agg_df[new_name].mean(), inplace=True)

agg_col_names = [
    new_name
    for aggs in FINAL_AGG_CONFIG.values()
    for new_name in aggs.keys()
]

FINAL_FEATURES_TEST = FINAL_FEATURES + agg_col_names
"""

# ── 3. Armar X_test ───────────────────────────────────────────────────────────
X_test = df_test[FINAL_FEATURES].copy()

for col in FINAL_CAT_FEATURES:
    if col in X_test.columns:
        X_test[col] = X_test[col].fillna(-1).astype(int)

"""
# ── 4. Reentrenar sobre todo el train ────────────────────────────────────────
X_train_full = df_fe[FINAL_FEATURES + FINAL_AUX_COLS].copy()

for group_col, aggs in FINAL_AGG_CONFIG.items():
    agg_kwargs = {
        new_name: pd.NamedAgg(column=src_col, aggfunc=func)
        for new_name, (src_col, func) in aggs.items()
    }
    agg_df = df_fe.groupby(group_col).agg(**agg_kwargs).reset_index()
    X_train_full = X_train_full.merge(agg_df, on=group_col, how='left')

# Solo dropear las aux_cols, no los group_cols que son features del modelo
cols_to_drop = [col for col in FINAL_AUX_COLS if col not in FINAL_FEATURES_TEST]
X_train_full = X_train_full.drop(columns=cols_to_drop, errors='ignore')
X_train_full = X_train_full[FINAL_FEATURES_TEST]

for col in FINAL_CAT_FEATURES:
    if col in X_train_full.columns:
        X_train_full[col] = X_train_full[col].fillna(-1).astype(int)

model_final = lgb.LGBMClassifier(**BEST_PARAMS, random_state=SEED)
model_final.fit(
    X_train_full, y,
    categorical_feature=FINAL_CAT_FEATURES or 'auto',
    callbacks=[lgb.log_evaluation(period=-1)]
)
"""
# ── 4. Reentrenar sobre todo el train ────────────────────────────────────────
X_train_full = df_fe[FINAL_FEATURES + FINAL_AUX_COLS].copy()

# Target Encoding fitteado sobre TODO el train (sin CV, porque no hay fold de val)
te_final = TargetEncoder(smooth=FINAL_TE_SMOOTHING, target_type='continuous', random_state=SEED)
te_cols_encoded = [c + '_te' for c in FINAL_TE_COLS]
X_train_full[te_cols_encoded] = te_final.fit_transform(X_train_full[FINAL_TE_COLS], y)

# Dropear aux_cols que no son features del modelo
cols_to_drop = [col for col in FINAL_AUX_COLS if col not in FINAL_FEATURES]
X_train_full = X_train_full.drop(columns=cols_to_drop, errors='ignore')

for col in FINAL_CAT_FEATURES:
    if col in X_train_full.columns:
        X_train_full[col] = X_train_full[col].fillna(-1).astype(int)

model_final = lgb.LGBMClassifier(**BEST_PARAMS, random_state=SEED)
model_final.fit(
    X_train_full, y,
    categorical_feature=FINAL_CAT_FEATURES or 'auto',
    callbacks=[lgb.log_evaluation(period=-1)]
)

# ── 5. Aplicar el mismo TE a test ─────────────────────────────────────────────
X_test = df_test[FINAL_FEATURES + FINAL_AUX_COLS].copy()
X_test[te_cols_encoded] = te_final.transform(X_test[FINAL_TE_COLS])  # solo transform, no fit
X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

for col in FINAL_CAT_FEATURES:
    if col in X_test.columns:
        X_test[col] = X_test[col].fillna(-1).astype(int)

# ── 6. Predecir y generar submission ──────────────────────────────────────────
test_proba = model_final.predict_proba(X_test)

submission = pd.DataFrame({
    'PetID': test['PetID'],
    'AdoptionSpeed': test_proba.argmax(axis=1)
})
submission.to_csv('submission.csv', index=False)
print(submission['AdoptionSpeed'].value_counts().sort_index())